# Recolly: Voice pipeline evaluation (Kaggle)

```
audio ──[ASR: PhoWhisper]──► transcript ──[Parser: Ollama LLM]──► todo_draft
        Tier 1: WER, KR, RTF        Tier 2: accuracy, F1, deadline
        └──────────── Tier 3: Task Success Rate, Δ_ASR ───────────┘
```

Kaggle-only copy of `voice_eval.ipynb` (use that one for local CPU runs).

**Before running**
1. *Datasets → New Dataset* → upload `recolly_voice_eval_dataset.zip` (contains `voice_eval.jsonl` + 62 `.m4a`). Keep it **Private**.
2. *File → Import notebook* → upload this file.
3. *Add Input* → that dataset. *Session options* → Accelerator **GPU T4 x2**, Internet **On**.
4. *Run All* (or *Save Version → Save & Run All* to run in the background, up to 12 h).

**Output**: `/kaggle/working/outputs/` + `/kaggle/working/voice_eval_outputs.zip` (download it, unzip into `server/notebooks/outputs/`).

**Resume after a timeout**: add the stopped version's *Output* as an extra Input. Section 0 copies the old
`*_results.jsonl` into `outputs/`, so finished (model, clip) pairs are skipped.

⚠️ Quality metrics transfer between machines; **speed metrics (RTF, latency) do not**: Kaggle has a GPU, the server does not.

## 0. Config

In [ ]:
import glob
import json
import math
import os
import re
import shutil
import subprocess
import time
import unicodedata
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
assert INPUT_DIR.exists(), "This notebook is for Kaggle. Run voice_eval.ipynb on your machine instead."

# --- What to run. Shrink these lists if the session gets close to the 12 h limit. ---
ASR_MODELS = ["vinai/PhoWhisper-small", "vinai/PhoWhisper-medium", "vinai/PhoWhisper-large"]
LLM_MODELS = ["qwen2.5:3b", "qwen2.5:7b", "gemma3:4b", "qwen3:4b"]
# Noise sweep cost: 60 clips × 3 noise types × 4 SNR = 720 transcriptions per model.
NOISY_ASR_MODELS = ["vinai/PhoWhisper-small"]

found = sorted(INPUT_DIR.rglob("voice_eval.jsonl"))
assert found, "Add the private dataset that contains voice_eval.jsonl as notebook Input"
DATA_DIR = found[0].parent
AUDIO_DIR = INPUT_DIR  # audio files are searched recursively under every Input
OUTPUT_DIR = WORKING_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLLAMA_URL = "http://localhost:11434"
CACHE_FILE_NAMES = ["asr_results.jsonl", "asr_noisy_results.jsonl", "parser_results.jsonl"]


def resume_previous_outputs() -> None:
    """Copy result caches from an earlier run (added as Input) into OUTPUT_DIR so finished work is skipped."""
    for file_name in CACHE_FILE_NAMES:
        target = OUTPUT_DIR / file_name
        previous = sorted(INPUT_DIR.rglob(file_name))
        if not target.exists() and previous:
            shutil.copy(previous[0], target)
            print(f"resumed {file_name} from {previous[0]}")


resume_previous_outputs()
print(f"DATA_DIR={DATA_DIR}\nASR={ASR_MODELS}\nNOISY_ASR={NOISY_ASR_MODELS}\nLLM={LLM_MODELS}")

## 1. Setup: GPU check, ffmpeg, Ollama

In [ ]:
def run_shell(command: str) -> None:
    """Run a shell command and stop the notebook if it fails."""
    print("$", command)
    subprocess.run(command, shell=True, check=True)


run_shell("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
if shutil.which("ffmpeg") is None:
    run_shell("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
if shutil.which("ollama") is None:
    run_shell("apt-get -qq install -y zstd > /dev/null")  # the installer unpacks a .tar.zst
    run_shell("curl -fsSL https://ollama.com/install.sh | sh")

assert shutil.which("ffmpeg"), "ffmpeg is required to decode m4a"
assert shutil.which("ollama"), "Ollama install failed: is Internet turned on in Session options?"

In [ ]:
def is_ollama_running() -> bool:
    """Return True when the Ollama HTTP server answers."""
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).ok
    except requests.RequestException:
        return False


def start_ollama_server() -> None:
    """Start `ollama serve` in the background (if needed) and wait until it is ready."""
    if is_ollama_running():
        print("Ollama already running")
        return
    log_file = open(OUTPUT_DIR / "ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_ollama_running():
            print("Ollama started")
            return
        time.sleep(1)
    raise RuntimeError(f"Ollama did not start, see {OUTPUT_DIR / 'ollama.log'}")


def pull_ollama_model(model_name: str) -> None:
    """Download an Ollama model (no-op if it is already present)."""
    start = time.perf_counter()
    response = requests.post(f"{OLLAMA_URL}/api/pull", json={"model": model_name, "stream": False}, timeout=3600)
    response.raise_for_status()
    print(f"pulled {model_name} in {time.perf_counter() - start:.0f}s")


start_ollama_server()
for llm_model in LLM_MODELS:
    pull_ollama_model(llm_model)

## 2. Data

One line of `voice_eval.jsonl` = one example. `audio: null` → text-only example (only the parser is evaluated).
`tags` let you slice results (e.g. `relative_date`, `no_deadline`).

In [ ]:
def load_eval_examples(path: Path) -> list[dict]:
    """Read the JSONL eval set, skipping blank lines."""
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


AUDIO_INDEX = {path.name: path for path in sorted(AUDIO_DIR.rglob("*.m4a"))}  # scan /kaggle/input once


def find_audio_file(file_name: str) -> Path | None:
    """Find an audio file by name anywhere under AUDIO_DIR."""
    return AUDIO_INDEX.get(file_name)


examples = load_eval_examples(DATA_DIR / "voice_eval.jsonl")
missing_audio = [e["audio"] for e in examples if e["audio"] and not find_audio_file(e["audio"])]
assert not missing_audio, f"{len(missing_audio)} audio files missing from the dataset: {missing_audio}"

print(f"{len(examples)} examples, {sum(1 for e in examples if e['audio'])} with audio")
pd.DataFrame([{"id": e["id"], "audio": e["audio"], "text": e["transcript_gold"], "tags": ", ".join(e["tags"])}
              for e in examples])

## 3. Text normalization — why `"ộ" != "ộ"`

Vietnamese diacritics can be stored *composed* (NFC, 1 code point) or *decomposed* (NFD, base letter + combining marks).
They look identical but compare as different strings → WER would count them as errors. Always normalize to NFC first.

In [ ]:
def normalize_vietnamese_text(text: str) -> str:
    """NFC-normalize, lowercase and strip punctuation so transcripts can be compared fairly."""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


composed = "ộ"
decomposed = unicodedata.normalize("NFD", composed)
print("equal before normalizing:", composed == decomposed, "| code points:", len(composed), "vs", len(decomposed))
print("equal after normalizing: ", normalize_vietnamese_text(composed) == normalize_vietnamese_text(decomposed))
print(normalize_vietnamese_text("Ngày mai, 9 GIỜ nộp báo cáo!"))

## 4. Metrics — ✍️ YOUR TURN

Each function = one formula. Replace `raise NotImplementedError` with your implementation, then run the test cell below.
The pipeline (sections 5–7) still runs while metrics are TODO; their columns just show `TODO`.

Suggested order: `accuracy` → `real_time_factor` → `token_f1` → `keyword_recall` → `macro_f1` → `mean_absolute_error_hours` → `confidence_interval_95` → `word_error_rate` (the hardest, worth it).

In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    """Return numerator / denominator, defining 0/0 (or x/0) as 0."""
    return numerator / denominator if denominator else 0.0


def accuracy(predictions: list, labels: list) -> float:
    """Fraction of exact matches.

    accuracy = (1/n) * Σ_i 1[ŷ_i == y_i]
    """
    correct_count = sum(1 for predicted, label in zip(predictions, labels) if predicted == label)
    return safe_divide(correct_count, len(labels))


def f1_from_precision_recall(precision: float, recall: float) -> float:
    """Harmonic mean of precision and recall.

    F1 = 2·P·R / (P + R)   (define 0/0 = 0)
    """
    return safe_divide(2 * precision * recall, precision + recall)


def macro_f1(predictions: list, labels: list) -> float:
    """Unweighted mean of per-class F1 over every class seen in predictions or labels.

    P_c = TP_c / (TP_c + FP_c),  R_c = TP_c / (TP_c + FN_c)   (define 0/0 = 0)
    F1_c = 2·P_c·R_c / (P_c + R_c)
    macroF1 = (1/|C|) * Σ_c F1_c
    """
    pairs = list(zip(predictions, labels))
    classes = set(predictions) | set(labels)
    per_class_f1 = []
    for current_class in classes:
        true_positive = sum(1 for predicted, label in pairs if predicted == current_class and label == current_class)
        false_positive = sum(1 for predicted, label in pairs if predicted == current_class and label != current_class)
        false_negative = sum(1 for predicted, label in pairs if predicted != current_class and label == current_class)
        precision = safe_divide(true_positive, true_positive + false_positive)
        recall = safe_divide(true_positive, true_positive + false_negative)
        per_class_f1.append(f1_from_precision_recall(precision, recall))
    return safe_divide(sum(per_class_f1), len(per_class_f1))


def token_f1(predicted_text: str, reference_text: str) -> float:
    """Bag-of-tokens F1 between two normalized texts (tokens = Vietnamese syllables).

    overlap = |Counter(pred) ∩ Counter(ref)|
    P = overlap / |pred|,  R = overlap / |ref|,  F1 = 2PR / (P + R)
    """
    predicted_tokens = normalize_vietnamese_text(predicted_text).split()
    reference_tokens = normalize_vietnamese_text(reference_text).split()
    overlap = sum((Counter(predicted_tokens) & Counter(reference_tokens)).values())
    precision = safe_divide(overlap, len(predicted_tokens))
    recall = safe_divide(overlap, len(reference_tokens))
    return f1_from_precision_recall(precision, recall)


def word_error_rate(hypothesis: str, reference: str) -> float:
    """Syllable-level WER via Levenshtein distance.

    WER = (S + D + I) / N,  N = number of reference syllables
    d[i][j] = edit distance between ref[:i] and hyp[:j]
    d[i][j] = min(d[i-1][j] + 1,                         # deletion
                  d[i][j-1] + 1,                         # insertion
                  d[i-1][j-1] + (ref[i-1] != hyp[j-1]))  # substitution / match
    """
    reference_tokens = normalize_vietnamese_text(reference).split()
    hypothesis_tokens = normalize_vietnamese_text(hypothesis).split()
    rows, columns = len(reference_tokens) + 1, len(hypothesis_tokens) + 1

    # d[i][0] = i (delete all i reference tokens), d[0][j] = j (insert all j hypothesis tokens)
    distance = [[0] * columns for _ in range(rows)]
    for i in range(rows):
        distance[i][0] = i
    for j in range(columns):
        distance[0][j] = j

    for i in range(1, rows):
        for j in range(1, columns):
            substitution_cost = int(reference_tokens[i - 1] != hypothesis_tokens[j - 1])
            distance[i][j] = min(distance[i - 1][j] + 1,
                                 distance[i][j - 1] + 1,
                                 distance[i - 1][j - 1] + substitution_cost)

    return safe_divide(distance[-1][-1], len(reference_tokens))


def keyword_recall(transcript: str, keywords: list[str]) -> float:
    """Fraction of important keywords that appear in the transcript (after normalization).

    KR = #keywords found / #keywords
    Matches whole syllables only, so "thi" does not match inside "thích".
    """
    padded_transcript = f" {normalize_vietnamese_text(transcript)} "
    found_count = sum(1 for keyword in keywords if f" {normalize_vietnamese_text(keyword)} " in padded_transcript)
    return safe_divide(found_count, len(keywords))


def real_time_factor(processing_seconds: float, audio_seconds: float) -> float:
    """How many seconds of compute per second of audio (< 1 means faster than real time).

    RTF = t_processing / t_audio
    """
    return safe_divide(processing_seconds, audio_seconds)


def mean_absolute_error_hours(predicted_times: list[datetime], true_times: list[datetime]) -> float:
    """Average absolute deadline error in hours.

    MAE_h = (1/m) * Σ_i |t̂_i − t_i| / 3600s
    """
    total_hours = sum(abs((predicted - true).total_seconds()) / 3600
                      for predicted, true in zip(predicted_times, true_times))
    return safe_divide(total_hours, len(true_times))


def confidence_interval_95(proportion: float, sample_size: int) -> tuple[float, float]:
    """Normal-approximation 95% confidence interval for a proportion.

    p ± 1.96 · sqrt(p(1 − p) / n)
    """
    margin = 1.96 * math.sqrt(proportion * (1 - proportion) / sample_size)
    return proportion - margin, proportion + margin

In [ ]:
def check_metric(name: str, test) -> None:
    """Run one metric test and print ✅ / ❌ / ⬜ TODO."""
    try:
        test()
        print(f"✅ {name}")
    except NotImplementedError:
        print(f"⬜ {name}: TODO")
    except AssertionError as error:
        print(f"❌ {name}: {error}")


def close(a, b) -> bool:
    """Float comparison with a small tolerance."""
    return math.isclose(a, b, abs_tol=1e-4)


def test_accuracy():
    assert close(accuracy(["a", "b", "c"], ["a", "b", "d"]), 2 / 3), "expected 2/3"

def test_macro_f1():
    assert close(macro_f1(["a", "a", "b"], ["a", "b", "b"]), 2 / 3), "expected 2/3 (both classes F1 = 2/3)"
    assert close(macro_f1(["a", "a"], ["a", "a"]), 1.0), "perfect predictions → 1.0"

def test_token_f1():
    assert close(token_f1("Nộp bài tập lớn", "nộp bài tập lớn cấu trúc dữ liệu"), 2 / 3), "P=1, R=4/8 → 2/3"
    assert close(token_f1("abc", "xyz"), 0.0), "no overlap → 0"

def test_word_error_rate():
    assert close(word_error_rate("ngày mai chín giờ gian nộp", "ngày mai chín giờ sáng nộp"), 1 / 6), "1 substitution / 6"
    assert close(word_error_rate("a b", "a b c"), 1 / 3), "1 deletion / 3"
    assert close(word_error_rate("a x b c", "a b c"), 1 / 3), "1 insertion / 3"
    assert close(word_error_rate("Ngày MAI!", "ngày mai"), 0.0), "normalize first"

def test_keyword_recall():
    assert close(keyword_recall("tối mai 8 giờ ôn thi giải tích", ["tối mai", "thi", "Giải Tích", "vật lý"]), 3 / 4)

def test_real_time_factor():
    assert close(real_time_factor(14, 4), 3.5), "the CPU demo: 14s for 4s of audio"

def test_mean_absolute_error_hours():
    t = datetime(2026, 11, 10, 8, 0)
    assert close(mean_absolute_error_hours([t, t], [t + timedelta(hours=2), t]), 1.0)

def test_confidence_interval_95():
    low, high = confidence_interval_95(0.8, 50)
    assert close(low, 0.8 - 0.110873) and close(high, 0.8 + 0.110873), "0.8 ± 0.1109"


for metric_test in [test_accuracy, test_real_time_factor, test_token_f1, test_keyword_recall, test_macro_f1,
                    test_mean_absolute_error_hours, test_confidence_interval_95, test_word_error_rate]:
    check_metric(metric_test.__name__.removeprefix("test_"), metric_test)

## 5. Run ASR (PhoWhisper)

Results are cached in `outputs/asr_results.jsonl`, so re-running skips work already done.

### The whole pipeline at a glance

```
                        ┌──────────── optional (section 5b) ────────────┐
                        │  + noise (white / pink / babble / real file)  │
                        │    mixed = signal + α·noise   at SNR 20…0 dB  │
                        └───────────────────────┬───────────────────────┘
 School.m4a                                     ▼
    │ ① ffmpeg decode → 16 kHz mono               waveform (75093,)        ← 4.69 s × 16000 samples
    ▼                                             │
 ┌──────────────── self.pipe(...) = transformers pipeline ────────────────────────────────────┐
 │  ② feature extractor   STFT 25 ms / 10 ms → 80 Mel filters → log   log-Mel (1, 80, 3000)  │  fixed formula
 │  ③ ENCODER  (Transformer "ears")                                    states  (1, 1500, 768) │  ┐ PhoWhisper
 │  ④ DECODER  (Transformer "writer"), one token at a time,                                   │  │ = the only
 │             primed with <|vi|><|transcribe|>                         token ids (22 tokens)  │  ┘ trained part
 │  ⑤ tokenizer: ids → UTF-8 bytes → text                                                     │  lookup table
 └────────────────────────────────────────────────────────────────────────────────────────────┘
    ▼
 transcript  "cặc lịch mời học trong trường lúc sáu giờ sáng."
    │                                             ── Tier 1 (ASR):    WER · Keyword Recall · RTF
    ▼
 Ollama LLM + JSON schema  (section 6)
    ▼
 todo_draft  {title, category, priority, deadline_at}
                                                  ── Tier 2 (parser): accuracy · macro-F1 · deadline MAE · title token-F1
                                                  ── Tier 3 (end-to-end): Task Success Rate · Δ_ASR = TSR(gold) − TSR(ASR)
```

| step | object in code | what it does |
|---|---|---|
| ① | `load_waveform()` (5b) / pipeline's own ffmpeg (5) | file → numbers in [-1, 1] |
| ② | `pipe.feature_extractor` | waveform → log-Mel spectrogram |
| ③ ④ | `pipe.model` | log-Mel → token ids (this is what "PhoWhisper" really is) |
| ⑤ | `pipe.tokenizer` | token ids → text |

`AsrRunner.transcribe(path)` runs ①–⑤; `AsrRunner.transcribe_waveform(array)` starts at ② so we can add noise to the waveform first.

In [ ]:
import torch
import transformers
from transformers import pipeline


def audio_duration_seconds(audio_path: Path) -> float:
    """Read audio length with ffprobe."""
    output = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(audio_path)],
        capture_output=True, text=True, check=True)
    return float(output.stdout.strip())


def load_jsonl_cache(path: Path) -> list[dict]:
    """Read cached result rows (empty list if the file does not exist yet)."""
    return load_eval_examples(path) if path.exists() else []


def append_jsonl(path: Path, row: dict) -> None:
    """Append one result row to a JSONL cache file."""
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")


ASR_SAMPLE_RATE = 16_000  # Whisper models expect 16 kHz mono audio


class AsrRunner:
    """Holds one PhoWhisper model and transcribes audio files with it."""

    def __init__(self, model_name: str):
        """Load the model on GPU (fp16) when available, otherwise CPU (fp32)."""
        self.model_name = model_name
        use_gpu = torch.cuda.is_available()
        dtype = torch.float16 if use_gpu else torch.float32
        major_version = int(transformers.__version__.split(".")[0])
        dtype_argument = {"dtype": dtype} if major_version >= 5 else {"torch_dtype": dtype}
        self.pipe = pipeline("automatic-speech-recognition", model=model_name,
                             device=0 if use_gpu else "cpu", chunk_length_s=30, **dtype_argument)

    def transcribe(self, audio_path: Path) -> tuple[str, float]:
        """Return (text, processing seconds) for one audio file, forcing Vietnamese."""
        start = time.perf_counter()
        output = self.pipe(str(audio_path), generate_kwargs={"language": "vi", "task": "transcribe"})
        return output["text"].strip(), time.perf_counter() - start

    def transcribe_waveform(self, waveform) -> tuple[str, float]:
        """Return (text, processing seconds) for a 16 kHz mono waveform already in memory (e.g. noisy mix)."""
        start = time.perf_counter()
        output = self.pipe({"raw": waveform, "sampling_rate": ASR_SAMPLE_RATE},
                           generate_kwargs={"language": "vi", "task": "transcribe"})
        return output["text"].strip(), time.perf_counter() - start


ASR_CACHE = OUTPUT_DIR / "asr_results.jsonl"
audio_examples = [e for e in examples if e["audio"]]

for asr_model in ASR_MODELS:
    done_ids = {r["id"] for r in load_jsonl_cache(ASR_CACHE) if r["asr_model"] == asr_model}
    todo = [e for e in audio_examples if e["id"] not in done_ids]
    if not todo:
        print(f"{asr_model}: cached")
        continue
    runner = AsrRunner(asr_model)
    runner.transcribe(find_audio_file(todo[0]["audio"]))  # warm-up, not timed
    for example in todo:
        audio_path = find_audio_file(example["audio"])
        text, seconds = runner.transcribe(audio_path)
        append_jsonl(ASR_CACHE, {"asr_model": asr_model, "id": example["id"], "transcript": text,
                                 "processing_seconds": round(seconds, 3),
                                 "audio_seconds": round(audio_duration_seconds(audio_path), 3)})
        print(f"{asr_model} | {example['id']} | {text}")
    del runner
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

asr_results = load_jsonl_cache(ASR_CACHE)

## 5b. Noise robustness — how much does background noise hurt PhoWhisper?

We take every **clean** recording and add noise at a fixed **SNR** (signal-to-noise ratio), then re-run ASR.
Same sentence, same speaker, only the noise changes → any rise in WER is caused by the noise.

```
SNR_dB = 10 · log10(P_signal / P_noise)          P = (1/N) · Σ x_i²
α      = sqrt(P_signal / (P_noise · 10^(SNR_dB/10)))
mixed  = signal + α · noise
```

| noise type | built from | simulates |
|---|---|---|
| `white`  | Gaussian random samples (flat spectrum) | hiss, fan, cheap mic |
| `pink`   | white noise shaped to power ∝ 1/f | traffic, rain, engine rumble |
| `babble` | sum of **other clips in this eval set** (different sentences) | canteen, café, crowded corridor |
| `file:<name>` | any real ambient recording put in `VOICE_NOISE_DIR` (optional) | the real place you recorded |

SNR levels: **20 dB** (slightly noisy) → 10 → 5 → **0 dB** (noise as loud as the voice).
A few mixed clips are saved to `outputs/noisy_samples/` — **listen to them** to get a feel for what each SNR sounds like.

In [ ]:
import wave
import zlib

import numpy as np

SNR_LEVELS_DB = [20, 10, 5, 0]
NOISE_TYPES = ["white", "pink", "babble"]
BABBLE_TALKERS = 4  # how many other clips are summed into one babble track
# optional real ambient recordings: put them in a folder named "noise" inside the dataset
NOISE_DIR = next((p for p in INPUT_DIR.rglob("noise") if p.is_dir()), WORKING_DIR / "noise")
NOISY_ASR_CACHE = OUTPUT_DIR / "asr_noisy_results.jsonl"
NOISY_SAMPLES_DIR = OUTPUT_DIR / "noisy_samples"


def load_waveform(audio_path: Path) -> np.ndarray:
    """Decode any audio file to a 16 kHz mono float32 waveform in [-1, 1] using ffmpeg."""
    output = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", str(audio_path), "-f", "f32le", "-ac", "1", "-ar", str(ASR_SAMPLE_RATE), "-"],
        capture_output=True, check=True)
    return np.frombuffer(output.stdout, dtype=np.float32).copy()


def save_waveform(waveform: np.ndarray, path: Path) -> None:
    """Write a float waveform in [-1, 1] as a 16-bit PCM WAV file (so you can listen to it)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    pcm = (np.clip(waveform, -1, 1) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as file:
        file.setnchannels(1)
        file.setsampwidth(2)
        file.setframerate(ASR_SAMPLE_RATE)
        file.writeframes(pcm.tobytes())


def signal_power(waveform: np.ndarray) -> float:
    """Average power of a waveform.

    P = (1/N) * Σ_i x_i²
    """
    return float(np.mean(waveform.astype(np.float64) ** 2))


def measured_snr_db(signal: np.ndarray, noise: np.ndarray) -> float:
    """Signal-to-noise ratio in decibels.

    SNR_dB = 10 * log10(P_signal / P_noise)
    """
    return 10 * math.log10(signal_power(signal) / signal_power(noise))


def scale_noise_to_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Rescale noise so that signal vs. scaled noise has exactly the target SNR.

    α = sqrt(P_signal / (P_noise * 10^(SNR_dB / 10))),   returns α * noise
    """
    alpha = math.sqrt(signal_power(signal) / (signal_power(noise) * 10 ** (snr_db / 10)))
    return alpha * noise


def mix_at_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Add noise to a signal at the target SNR, then peak-normalize if the sum would clip.

    mixed = signal + α * noise;  if max|mixed| > 1: mixed = mixed / max|mixed|  (SNR unchanged)
    """
    mixed = signal + scale_noise_to_snr(signal, noise, snr_db)
    peak = float(np.max(np.abs(mixed)))
    return (mixed / peak if peak > 1 else mixed).astype(np.float32)


class NoiseGenerator:
    """Builds reproducible noise tracks (white, pink, babble, real files) of any length."""

    def __init__(self, clean_clips: dict[str, np.ndarray], gold_texts: dict[str, str], noise_files: list[Path]):
        """Keep the clean clips (babble source) and the optional real-noise recordings."""
        self.clean_clips = clean_clips
        self.gold_texts = gold_texts
        self.noise_waveforms = {f"file:{path.stem}": load_waveform(path) for path in noise_files}

    def available_noise_types(self) -> list[str]:
        """Synthetic types plus one type per real noise file."""
        return NOISE_TYPES + sorted(self.noise_waveforms)

    def make_noise(self, noise_type: str, length: int, example_id: str) -> np.ndarray | None:
        """Return a noise track of `length` samples, or None if this type cannot be built."""
        rng = np.random.default_rng(zlib.crc32(f"{example_id}|{noise_type}".encode()))  # same noise every run
        if noise_type == "white":
            return rng.standard_normal(length)
        if noise_type == "pink":
            return self._pink_noise(length, rng)
        if noise_type == "babble":
            return self._babble_noise(length, rng, example_id)
        return self._fit_length(self.noise_waveforms[noise_type], length, rng)

    @staticmethod
    def _pink_noise(length: int, rng: np.random.Generator) -> np.ndarray:
        """Pink noise: shape white noise in the frequency domain.

        |X(f)|² ∝ 1/f   ⇒   multiply each FFT bin of white noise by 1/sqrt(f)
        """
        spectrum = np.fft.rfft(rng.standard_normal(length))
        frequencies = np.fft.rfftfreq(length)
        frequencies[0] = frequencies[1]  # avoid dividing by zero at DC
        return np.fft.irfft(spectrum / np.sqrt(frequencies), n=length)

    def _babble_noise(self, length: int, rng: np.random.Generator, example_id: str) -> np.ndarray | None:
        """Crowd noise: sum of other clips (different sentence), each scaled to equal power.

        babble = Σ_k clip_k / sqrt(P_k)
        """
        own_text = self.gold_texts[example_id]
        candidates = [clip_id for clip_id in self.clean_clips
                      if clip_id != example_id and self.gold_texts[clip_id] != own_text]
        if not candidates:
            return None
        chosen = rng.choice(candidates, size=min(BABBLE_TALKERS, len(candidates)), replace=False)
        babble = np.zeros(length)
        for clip_id in chosen:
            clip = self._fit_length(self.clean_clips[clip_id], length, rng)
            babble += clip / math.sqrt(signal_power(clip))
        return babble

    @staticmethod
    def _fit_length(waveform: np.ndarray, length: int, rng: np.random.Generator) -> np.ndarray:
        """Loop a waveform until it is long enough, then cut a random window of `length` samples."""
        repeats = math.ceil((length + len(waveform)) / len(waveform))
        looped = np.tile(waveform, repeats)
        start = int(rng.integers(0, len(waveform)))
        return looped[start:start + length].astype(np.float64)


# --- sanity check: the mixer must hit the target SNR exactly ---
test_rng = np.random.default_rng(0)
test_signal = np.sin(np.linspace(0, 200 * np.pi, ASR_SAMPLE_RATE)) * 0.3
for target_snr in SNR_LEVELS_DB:
    scaled = scale_noise_to_snr(test_signal, test_rng.standard_normal(ASR_SAMPLE_RATE), target_snr)
    assert math.isclose(measured_snr_db(test_signal, scaled), target_snr, abs_tol=1e-6)
print("✅ scale_noise_to_snr hits the target SNR")

# --- build noise + run ASR on every (clip, noise type, SNR) ---
clean_clips = {e["id"]: load_waveform(find_audio_file(e["audio"])) for e in audio_examples}
noise_files = sorted(p for p in NOISE_DIR.glob("*") if p.suffix.lower() in {".wav", ".m4a", ".mp3", ".ogg", ".flac"}) \
    if NOISE_DIR.exists() else []
noise_generator = NoiseGenerator(clean_clips, {e["id"]: e["transcript_gold"] for e in examples}, noise_files)
print("noise types:", noise_generator.available_noise_types())

for asr_model in NOISY_ASR_MODELS:
    done = {(r["id"], r["noise_type"], r["snr_db"]) for r in load_jsonl_cache(NOISY_ASR_CACHE)
            if r["asr_model"] == asr_model}
    todo = [(e, noise_type, snr) for e in audio_examples for noise_type in noise_generator.available_noise_types()
            for snr in SNR_LEVELS_DB if (e["id"], noise_type, snr) not in done]
    if not todo:
        print(f"{asr_model}: cached")
        continue
    runner = AsrRunner(asr_model)
    runner.transcribe_waveform(clean_clips[todo[0][0]["id"]])  # warm-up, not timed
    for example, noise_type, snr in todo:
        clean = clean_clips[example["id"]]
        noise = noise_generator.make_noise(noise_type, len(clean), example["id"])
        if noise is None:
            continue  # e.g. babble needs at least one other clip
        mixed = mix_at_snr(clean, noise, snr)
        if example["id"] == audio_examples[0]["id"]:
            save_waveform(mixed, NOISY_SAMPLES_DIR / f"{example['id']}_{noise_type.replace(':', '-')}_{snr}dB.wav")
        text, seconds = runner.transcribe_waveform(mixed)
        append_jsonl(NOISY_ASR_CACHE, {"asr_model": asr_model, "id": example["id"], "noise_type": noise_type,
                                       "snr_db": snr, "transcript": text, "processing_seconds": round(seconds, 3)})
        print(f"{asr_model} | {example['id']} | {noise_type:7} | {snr:>2} dB | {text}")
    del runner
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

noisy_asr_results = load_jsonl_cache(NOISY_ASR_CACHE)

# --- report: WER / keyword recall per noise type and SNR (clean = baseline from section 5) ---
gold_by_id = {e["id"]: e for e in examples}
noise_rows = [{"asr_model": r["asr_model"], "noise_type": r["noise_type"], "snr_db": r["snr_db"],
               "WER": word_error_rate(r["transcript"], gold_by_id[r["id"]]["transcript_gold"]),
               "KR": keyword_recall(r["transcript"], gold_by_id[r["id"]]["keywords"])}
              for r in noisy_asr_results]
clean_rows = [{"asr_model": r["asr_model"], "snr_db": "clean",
               "WER": word_error_rate(r["transcript"], gold_by_id[r["id"]]["transcript_gold"]),
               "KR": keyword_recall(r["transcript"], gold_by_id[r["id"]]["keywords"])}
              for r in asr_results]
if noise_rows:
    noise_table = pd.DataFrame(noise_rows)
    clean_table = pd.DataFrame(clean_rows).groupby("asr_model")[["WER", "KR"]].mean()
    column_order = ["clean"] + SNR_LEVELS_DB
    for metric in ["WER", "KR"]:
        pivot = noise_table.pivot_table(index=["asr_model", "noise_type"], columns="snr_db", values=metric, aggfunc="mean")
        pivot["clean"] = [clean_table.loc[model, metric] for model, _ in pivot.index]
        print(f"{metric} by noise type × SNR (dB)   — lower WER / higher KR is better")
        display(pivot[[c for c in column_order if c in pivot.columns]].round(3))

## 6. Run parser (Ollama + JSON schema)

**Baseline**: the LLM computes `deadline_at` itself. Later experiment: let the LLM output *relative* time
(`weekday`, `day_offset`, `time`) and compute the date in Python — then compare the numbers.

Each LLM is run on two inputs: the **gold transcript** and every **ASR transcript** → lets us compute Δ_ASR.

In [ ]:
VIETNAMESE_WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]

TODO_DRAFT_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "assignment", "exam", "personal", "other"]},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "deadline_at": {"type": ["string", "null"], "description": "ISO 8601 with +07:00 offset, or null"},
    },
    "required": ["title", "category", "priority", "deadline_at"],
}


def describe_reference_time(reference_time: str) -> str:
    """Turn an ISO timestamp into 'Thứ Ba, 2026-11-10 14:00 (+07:00)' for the prompt."""
    moment = datetime.fromisoformat(reference_time)
    return f"{VIETNAMESE_WEEKDAYS[moment.weekday()]}, {moment:%Y-%m-%d %H:%M} (+07:00)"


def build_system_prompt(reference_time: str) -> str:
    """System prompt telling the LLM how to turn one utterance into a todo."""
    return (
        "Bạn trích xuất MỘT việc cần làm từ câu nói tiếng Việt của sinh viên.\n"
        f"Thời điểm hiện tại: {describe_reference_time(reference_time)}.\n"
        "- title: ngắn gọn, viết hoa chữ đầu, không chứa thời gian.\n"
        "- category: study (học/ôn), assignment (bài tập/nộp/báo cáo), exam (thi/kiểm tra), personal, other.\n"
        "- priority: high nếu có 'gấp', 'quan trọng'; low nếu việc vặt; còn lại medium.\n"
        "- deadline_at: ISO 8601 +07:00. Có ngày mà không có giờ thì dùng 23:59. Không nhắc thời gian thì null."
    )


class OllamaTodoParser:
    """Turns a transcript into a todo_draft dict using an Ollama model with structured output."""

    def __init__(self, model_name: str):
        """Remember which Ollama model to call."""
        self.model_name = model_name

    def parse(self, transcript: str, reference_time: str) -> tuple[dict | None, float, str | None]:
        """Return (draft or None, seconds, error message or None)."""
        payload = {
            "model": self.model_name,
            "stream": False,
            "format": TODO_DRAFT_SCHEMA,
            "options": {"temperature": 0},
            "messages": [{"role": "system", "content": build_system_prompt(reference_time)},
                         {"role": "user", "content": transcript}],
        }
        if self.model_name.startswith("qwen3"):
            payload["think"] = False  # thinking mode is slow and not needed here
        start = time.perf_counter()
        try:
            response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
            response.raise_for_status()
            draft = json.loads(response.json()["message"]["content"])
            return draft, time.perf_counter() - start, None
        except (requests.RequestException, json.JSONDecodeError, KeyError) as error:
            return None, time.perf_counter() - start, str(error)


def parser_inputs() -> list[tuple[str, dict, str]]:
    """List (source, example, transcript): gold text for all examples + every ASR output."""
    by_id = {e["id"]: e for e in examples}
    inputs = [("gold", e, e["transcript_gold"]) for e in examples]
    inputs += [(f"asr:{r['asr_model']}", by_id[r["id"]], r["transcript"]) for r in asr_results]
    return inputs


PARSER_CACHE = OUTPUT_DIR / "parser_results.jsonl"

for llm_model in LLM_MODELS:
    done = {(r["source"], r["id"]) for r in load_jsonl_cache(PARSER_CACHE) if r["llm_model"] == llm_model}
    todo = [(s, e, t) for s, e, t in parser_inputs() if (s, e["id"]) not in done]
    if not todo:
        print(f"{llm_model}: cached")
        continue
    parser = OllamaTodoParser(llm_model)
    parser.parse("test", examples[0]["reference_time"])  # warm-up: loads the model into memory
    for source, example, transcript in todo:
        draft, seconds, error = parser.parse(transcript, example["reference_time"])
        append_jsonl(PARSER_CACHE, {"llm_model": llm_model, "source": source, "id": example["id"],
                                    "transcript": transcript, "draft": draft,
                                    "processing_seconds": round(seconds, 3), "error": error})
        print(f"{llm_model} | {source:28} | {example['id']} | {json.dumps(draft, ensure_ascii=False)}")

parser_results = load_jsonl_cache(PARSER_CACHE)

## 7. Report

In [ ]:
from datetime import timezone


def safe_metric(function, *args):
    """Call a metric, returning 'TODO' while it is not implemented yet."""
    try:
        return round(function(*args), 3)
    except NotImplementedError:
        return "TODO"


VIETNAM_TIMEZONE = timezone(timedelta(hours=7))


def parse_deadline(value: str | None) -> datetime | None:
    """Parse an ISO 8601 deadline; None when missing or malformed (LLMs emit things like '+07:0:00').

    A deadline without an offset is assumed to be Vietnam time (+07:00), so it can be subtracted safely.
    """
    if not isinstance(value, str) or not value:
        return None
    try:
        moment = datetime.fromisoformat(value)
    except ValueError:
        return None
    return moment if moment.tzinfo else moment.replace(tzinfo=VIETNAM_TIMEZONE)


def is_invalid_deadline(value) -> bool:
    """True when the model returned a deadline string that cannot be parsed."""
    return value is not None and parse_deadline(value) is None


def same_deadline_day(predicted: str | None, expected: str | None) -> bool:
    """True when both deadlines are null, or both fall on the same calendar day (malformed = wrong)."""
    if predicted is None or expected is None:
        return predicted is None and expected is None
    predicted_moment, expected_moment = parse_deadline(predicted), parse_deadline(expected)
    if predicted_moment is None or expected_moment is None:
        return False
    return predicted_moment.date() == expected_moment.date()


def is_task_success(draft: dict | None, expected: dict) -> bool:
    """Headline metric: category right AND deadline on the right day AND title roughly right."""
    if draft is None:
        return False
    try:
        title_ok = token_f1(draft.get("title", ""), expected["title"]) >= 0.5
    except NotImplementedError:
        title_ok = True  # until token_f1 exists, judge only category + deadline
    return (draft.get("category") == expected["category"]
            and same_deadline_day(draft.get("deadline_at"), expected["deadline_at"])
            and title_ok)


by_id = {e["id"]: e for e in examples}

# --- Tier 1: ASR ---
asr_rows = []
for asr_model in ASR_MODELS:
    rows = [r for r in asr_results if r["asr_model"] == asr_model]
    if not rows:
        continue
    asr_rows.append({
        "asr_model": asr_model, "n": len(rows),
        "WER": safe_metric(lambda: sum(word_error_rate(r["transcript"], by_id[r["id"]]["transcript_gold"])
                                       for r in rows) / len(rows)),
        "keyword_recall": safe_metric(lambda: sum(keyword_recall(r["transcript"], by_id[r["id"]]["keywords"])
                                                  for r in rows) / len(rows)),
        "RTF (this machine)": safe_metric(lambda: sum(real_time_factor(r["processing_seconds"], r["audio_seconds"])
                                                      for r in rows) / len(rows)),
    })
print("Tier 1 — ASR")
display(pd.DataFrame(asr_rows))

# --- Tier 2 + 3: parser and end-to-end ---
parser_rows = []
for (llm_model, source), group in pd.DataFrame(parser_results).groupby(["llm_model", "source"]):
    rows = group.to_dict("records")
    expected = [by_id[r["id"]]["expected"] for r in rows]
    drafts = [r["draft"] or {} for r in rows]
    successes = [is_task_success(r["draft"], exp) for r, exp in zip(rows, expected)]
    success_rate = sum(successes) / len(successes)
    deadline_pairs = [(parse_deadline(d.get("deadline_at")), parse_deadline(e["deadline_at"]))
                      for d, e in zip(drafts, expected)]
    with_deadline = [(predicted, true) for predicted, true in deadline_pairs if predicted and true]
    parser_rows.append({
        "llm_model": llm_model, "source": source, "n": len(rows),
        "schema_valid": round(sum(r["draft"] is not None for r in rows) / len(rows), 3),
        "category_acc": safe_metric(accuracy, [d.get("category") for d in drafts], [e["category"] for e in expected]),
        "category_macroF1": safe_metric(macro_f1, [d.get("category") for d in drafts], [e["category"] for e in expected]),
        "priority_acc": safe_metric(accuracy, [d.get("priority") for d in drafts], [e["priority"] for e in expected]),
        "deadline_day_acc": round(sum(same_deadline_day(d.get("deadline_at"), e["deadline_at"])
                                      for d, e in zip(drafts, expected)) / len(rows), 3),
        "deadline_MAE_h": safe_metric(lambda: mean_absolute_error_hours(
            [predicted for predicted, _ in with_deadline],
            [true for _, true in with_deadline])) if with_deadline else None,
        "deadline_invalid": sum(is_invalid_deadline(d.get("deadline_at")) for d in drafts),
        "title_tokenF1": safe_metric(lambda: sum(token_f1(d.get("title", ""), e["title"])
                                                 for d, e in zip(drafts, expected)) / len(rows)),
        "TASK_SUCCESS": round(success_rate, 3),
        "TSR_CI95": safe_metric(lambda: confidence_interval_95(success_rate, len(rows))[1] - success_rate),
        "latency_s (this machine)": round(group["processing_seconds"].mean(), 2),
    })
parser_table = pd.DataFrame(parser_rows)
print("Tier 2/3 — Parser & end-to-end   (TSR_CI95 = ± half-width)")
display(parser_table)

In [ ]:
# Δ_ASR: how much task success is lost because of ASR mistakes (audio examples only)
audio_ids = {e["id"] for e in audio_examples}
delta_rows = []
for llm_model in LLM_MODELS:
    rows = [r for r in parser_results if r["llm_model"] == llm_model and r["id"] in audio_ids]
    gold = [is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows if r["source"] == "gold"]
    for asr_model in ASR_MODELS:
        asr = [is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows if r["source"] == f"asr:{asr_model}"]
        if gold and asr:
            delta_rows.append({"llm_model": llm_model, "asr_model": asr_model,
                               "TSR_gold": sum(gold) / len(gold), "TSR_asr": sum(asr) / len(asr),
                               "Δ_ASR": sum(gold) / len(gold) - sum(asr) / len(asr)})
print("Δ_ASR = TSR(gold transcript) − TSR(ASR transcript)")
display(pd.DataFrame(delta_rows))

# Task success sliced by tag (gold transcripts) — shows WHERE each model fails
tag_rows = []
for r in parser_results:
    if r["source"] != "gold":
        continue
    for tag in by_id[r["id"]]["tags"]:
        tag_rows.append({"llm_model": r["llm_model"], "tag": tag,
                         "success": is_task_success(r["draft"], by_id[r["id"]]["expected"])})
print("Task success by tag (gold transcripts)")
display(pd.DataFrame(tag_rows).pivot_table(index="tag", columns="llm_model", values="success", aggfunc="mean").round(2))

In [ ]:
# Inspect failures one by one — reading errors teaches more than any average
failures = [
    {"llm": r["llm_model"], "source": r["source"], "id": r["id"], "transcript": r["transcript"],
     "pred_deadline": (r["draft"] or {}).get("deadline_at"), "exp_deadline": by_id[r["id"]]["expected"]["deadline_at"],
     "pred_cat": (r["draft"] or {}).get("category"), "exp_cat": by_id[r["id"]]["expected"]["category"],
     "pred_title": (r["draft"] or {}).get("title"), "error": r["error"]}
    for r in parser_results if not is_task_success(r["draft"], by_id[r["id"]]["expected"])
]
pd.set_option("display.max_colwidth", 60)
display(pd.DataFrame(failures))

## 8. Download results

In [ ]:
archive_path = shutil.make_archive(str(WORKING_DIR / "voice_eval_outputs"), "zip", OUTPUT_DIR)
print("Download from the Output tab:", archive_path)
for path in sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(f"  {path.name}: {sum(1 for _ in open(path, encoding='utf-8'))} rows")

In [ ]:
def mcnemar_exact_p_value(only_a_correct: int, only_b_correct: int) -> float:
      """Two-sided exact McNemar test on the discordant pairs.

      b = #(A right, B wrong),  c = #(A wrong, B right),  n = b + c
      Under H0 (both methods equally good) each discordant pair is a fair coin: b ~ Binomial(n, 0.5)
      p = min(1, 2 · Σ_{k=0}^{min(b,c)} C(n, k) · 0.5^n)
      """
      discordant = only_a_correct + only_b_correct
      if discordant == 0:
          return 1.0
      smaller = min(only_a_correct, only_b_correct)
      tail = sum(math.comb(discordant, k) for k in range(smaller + 1)) / 2 ** discordant
      return min(1.0, 2 * tail)

def task_success_by_id(results: list[dict], llm_model: str, source: str) -> dict[str, bool]:
      """Map example id → task success for one (LLM, transcript source) run."""
def task_success_by_id(results: list[dict], llm_model: str, source: str) -> dict[str, bool]:
    """Map example id → task success for one (LLM, transcript source) run."""
    return {r["id"]: is_task_success(r["draft"], by_id[r["id"]]["expected"])
            for r in results if r["llm_model"] == llm_model and r["source"] == source}


class PairedComparison:
    """Compares two runs on the sentences both of them processed (2×2 contingency table)."""

    def __init__(self, label_a: str, success_a: dict[str, bool], label_b: str, success_b: dict[str, bool]):
        """Keep only the ids present in both runs, so every row is a true pair."""
        self.label_a, self.label_b = label_a, label_b
        shared_ids = sorted(success_a.keys() & success_b.keys())
        self.pairs = [(success_a[i], success_b[i]) for i in shared_ids]

    def summary(self, alpha: float = 0.05) -> dict:
        """Counts of the 2×2 table, both success rates, the McNemar p-value and a plain-language verdict."""
        both = sum(a and b for a, b in self.pairs)
        only_a = sum(a and not b for a, b in self.pairs)
        only_b = sum(b and not a for a, b in self.pairs)
        neither = sum(not a and not b for a, b in self.pairs)
        n = len(self.pairs)
        p_value = mcnemar_exact_p_value(only_a, only_b)
        if p_value >= alpha:
            verdict = "no significant difference"
        else:
            verdict = f"{self.label_a if only_a > only_b else self.label_b} better"
        return {"A": self.label_a, "B": self.label_b, "n": n,
                "both_ok": both, "only_A_ok": only_a, "only_B_ok": only_b, "both_wrong": neither,
                "TSR_A": round(safe_divide(both + only_a, n), 3), "TSR_B": round(safe_divide(both + only_b, n), 3),
                "p_value": round(p_value, 4), "verdict": verdict}


# --- sanity checks against hand-computed values ---
assert math.isclose(mcnemar_exact_p_value(0, 6), 2 / 64), "6 discordant, all one way → 2·(1/64)"
assert math.isclose(mcnemar_exact_p_value(1, 9), 2 * 11 / 1024), "C(10,0)+C(10,1) = 11"
assert mcnemar_exact_p_value(5, 5) == 1.0 and mcnemar_exact_p_value(0, 0) == 1.0
print("✅ mcnemar_exact_p_value")

# --- which pairs to compare ---
run_names = sorted({r["llm_model"] for r in parser_results})
sources = sorted({r["source"] for r in parser_results})
comparisons = []  # (label_a, llm_a, source_a, label_b, llm_b, source_b)

# 1. old prompt vs resolve_deadline, same LLM and same transcript
for name in run_names:
    if f"{name}+py" in run_names:
        for source in sources:
            comparisons.append((name, name, source, f"{name}+py", f"{name}+py", source))

# 2. LLM vs LLM on gold transcripts (edit this list freely)
EXTRA_LLM_PAIRS = [("qwen2.5:7b", "gemma3:4b"), ("qwen2.5:7b", "qwen2.5:3b")]
for llm_model in ["qwen2.5:7b"]:          # was: for llm_model in LLM_MODELS:
    todo = [(s, e, t) for s, e, t in parser_inputs() if s == "gold" and (s, e["id"]) not in done]

# 3. ASR vs ASR for the same LLM: does a bigger Whisper change task success?
ASR_PAIRS = [("vinai/PhoWhisper-small", "vinai/PhoWhisper-large"), ("vinai/PhoWhisper-medium", "vinai/PhoWhisper-large")]
for name in run_names:
    for asr_a, asr_b in ASR_PAIRS:
        if f"asr:{asr_a}" in sources and f"asr:{asr_b}" in sources:
            comparisons.append((f"{name} | {asr_a.split('-')[-1]}", name, f"asr:{asr_a}",
                                f"{name} | {asr_b.split('-')[-1]}", name, f"asr:{asr_b}"))

paired_rows = []
for label_a, llm_a, source_a, label_b, llm_b, source_b in comparisons:
    comparison = PairedComparison(label_a, task_success_by_id(parser_results, llm_a, source_a),
                                  label_b, task_success_by_id(parser_results, llm_b, source_b))
    row = comparison.summary()
    row["source"] = source_a if source_a == source_b else f"{source_a} vs {source_b}"
    paired_rows.append(row)

print("McNemar paired test on TASK_SUCCESS   (only_A_ok / only_B_ok = the pairs that decide the test)")
display(pd.DataFrame(paired_rows)[["A", "B", "source", "n", "both_ok", "only_A_ok", "only_B_ok", "both_wrong",
                                   "TSR_A", "TSR_B", "p_value", "verdict"]])


## 9. Deadline experiments

Run in order: **9a** (+py: LLM reads time words, Python computes the date) → **9b** (calendar table in the prompt) → **9c** (paired McNemar head-to-head) → **9d** (what is still wrong).

### 9a. `resolve_deadline` (+py)

In [ ]:
import time
from datetime import date, datetime, timedelta, timezone
from datetime import time as clock_time

VIETNAM_TIMEZONE = timezone(timedelta(hours=7))
END_OF_DAY = clock_time(23, 59)

# The LLM only *reads* time expressions; Python does the calendar arithmetic.
TODO_DRAFT_SCHEMA_RELATIVE = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "assignment", "exam", "personal", "other"]},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "date_type": {"type": "string", "enum": ["none", "relative_days", "weekday", "calendar_date"],
                      "description": "relative_days: nay/mai/mốt/'N ngày nữa'/'N tuần nữa'; weekday: có nói thứ "
                                     "hoặc cuối tuần; calendar_date: có 'ngày N tháng M'; none: không nói ngày"},
        "day_offset": {"type": ["integer", "null"],
                       "description": "nay/tối nay=0, mai=1, mốt=2, 'ba ngày nữa'=3, 'hai tuần nữa'=14; null nếu không có"},
        "weekday": {"type": ["integer", "null"],
                    "description": "0=Thứ Hai, 1=Thứ Ba, ..., 5=Thứ Bảy, 6=Chủ Nhật hoặc 'cuối tuần'; null nếu không nói thứ"},
        "week_offset": {"type": "integer", "description": "1 nếu có 'tuần sau', còn lại 0"},
        "day_of_month": {"type": ["integer", "null"], "description": "chỉ khi nói rõ 'ngày N', còn lại null"},
        "month": {"type": ["integer", "null"], "description": "chỉ khi nói rõ 'tháng M', còn lại null"},
        "time": {"type": ["string", "null"],
                 "description": "giờ dạng HH:MM 24h ('bảy giờ tối'=19:00, 'tám giờ kém mười lăm tối'=19:45); null nếu không nói giờ"},
    },
    "required": ["title", "category", "priority", "date_type", "day_offset", "weekday", "week_offset",
                 "day_of_month", "month", "time"],
}


def parse_clock_time(text: str | None) -> clock_time | None:
    """Parse 'HH:MM' (24h) into a time; None when missing or malformed."""
    if not text:
        return None
    try:
        hours, minutes = (int(part) for part in text.strip().split(":")[:2])
        return clock_time(hours, minutes)
    except (ValueError, TypeError):
        return None


def date_from_day_offset(reference_day: date, day_offset: int) -> date:
    """Relative day count.

    date = reference_date + day_offset
    """
    return reference_day + timedelta(days=day_offset)


def date_from_weekday(reference_day: date, weekday: int, week_offset: int) -> date:
    """Named weekday, optionally in a later week (Monday = 0 ... Sunday = 6).

    monday = reference_date − reference_date.weekday()
    date   = monday + 7·week_offset + weekday
    if week_offset = 0 and date < reference_date: date += 7   (that day already passed this week)
    """
    monday = reference_day - timedelta(days=reference_day.weekday())
    target = monday + timedelta(days=7 * week_offset + weekday)
    if week_offset == 0 and target < reference_day:
        target += timedelta(days=7)
    return target


def date_from_day_and_month(reference_day: date, day_of_month: int, month: int | None) -> date | None:
    """Explicit 'ngày N tháng M': this year, or next year if that date already passed; None if invalid.

    Without a month, use the reference month (next month if that day already passed).
    """
    try:
        if month is None:
            target = reference_day.replace(day=day_of_month)
            if target < reference_day:
                next_month_first = (reference_day.replace(day=1) + timedelta(days=32)).replace(day=1)
                target = next_month_first.replace(day=day_of_month)
            return target
        target = date(reference_day.year, month, day_of_month)
        return target if target >= reference_day else date(reference_day.year + 1, month, day_of_month)
    except ValueError:  # e.g. 31/2 or day 31 in a 30-day month
        return None


def resolve_deadline(fields: dict, reference: datetime) -> datetime | None:
    """Turn the LLM's relative time fields into an absolute deadline (+07:00), or None if no time was said.

    date_type decides which fields are read; the others are ignored (small LLMs often put 0 instead of null):
      calendar_date → day_of_month (+ month)
      relative_days → reference_date + day_offset
      weekday       → monday + 7·week_offset + weekday
      none          → only a time: today if that time is still ahead, otherwise tomorrow; no time: no deadline
    Clock: the spoken time, or 23:59 when only a day was said.
    """
    reference = reference if reference.tzinfo else reference.replace(tzinfo=VIETNAM_TIMEZONE)
    reference = reference.astimezone(VIETNAM_TIMEZONE)
    reference_day = reference.date()
    clock = parse_clock_time(fields.get("time"))

    date_type = fields.get("date_type")

    if date_type == "calendar_date" and fields.get("day_of_month"):
        target_day = date_from_day_and_month(reference_day, fields["day_of_month"], fields.get("month") or None)
    elif date_type == "relative_days":
        target_day = date_from_day_offset(reference_day, fields.get("day_offset") or 0)
    elif date_type == "weekday" and fields.get("weekday") is not None and 0 <= fields["weekday"] <= 6:
        target_day = date_from_weekday(reference_day, fields["weekday"], fields.get("week_offset") or 0)
    elif clock is not None:
        today_at_clock = datetime.combine(reference_day, clock, VIETNAM_TIMEZONE)
        target_day = reference_day if today_at_clock > reference else reference_day + timedelta(days=1)
    else:
        return None  # no date and no time → task without deadline

    if target_day is None:
        return None
    return datetime.combine(target_day, clock or END_OF_DAY, VIETNAM_TIMEZONE)


# --- tests: the sentences the LLM got wrong, with the fields a correct reading would give ---
def check_resolve_deadline() -> None:
    """Assert resolve_deadline matches the eval labels (reference = Thứ Ba 2026-11-10 14:00)."""
    reference = datetime(2026, 11, 10, 14, 0, tzinfo=VIETNAM_TIMEZONE)
    blank = {"date_type": "none", "day_offset": None, "weekday": None, "week_offset": 0, "day_of_month": None, "month": None, "time": None}
    cases = [  # (sentence, fields, expected "YYYY-MM-DD HH:MM" or None)
        ("lúc sáu giờ sáng",                {"time": "06:00"},                  "2026-11-11 06:00"),
        ("lúc bảy giờ tối",                 {"time": "19:00"},                  "2026-11-10 19:00"),
        ("lúc mười hai giờ rưỡi",           {"time": "12:30"},                  "2026-11-11 12:30"),
        ("ngày mai nộp báo cáo",            {"date_type": "relative_days", "day_offset": 1},                  "2026-11-11 23:59"),
        ("ngày mốt kiểm tra",               {"date_type": "relative_days", "day_offset": 2},                  "2026-11-12 23:59"),
        ("tối nay làm bài tập",             {"date_type": "relative_days", "day_offset": 0},                  "2026-11-10 23:59"),
        ("sáng mai bảy giờ",                {"date_type": "relative_days", "day_offset": 1, "time": "07:00"}, "2026-11-11 07:00"),
        ("ba ngày nữa",                     {"date_type": "relative_days", "day_offset": 3},                  "2026-11-13 23:59"),
        ("hai tuần nữa",                    {"date_type": "relative_days", "day_offset": 14},                 "2026-11-24 23:59"),
        ("thứ tư thi kiến trúc máy tính",   {"date_type": "weekday", "weekday": 2},                     "2026-11-11 23:59"),
        ("thứ năm tuần này",                {"date_type": "weekday", "weekday": 3},                     "2026-11-12 23:59"),
        ("tối thứ sáu này tám giờ",         {"date_type": "weekday", "weekday": 4, "time": "20:00"},    "2026-11-13 20:00"),
        ("chủ nhật này chín giờ sáng",      {"date_type": "weekday", "weekday": 6, "time": "09:00"},    "2026-11-15 09:00"),
        ("cuối tuần này",                   {"date_type": "weekday", "weekday": 6},                     "2026-11-15 23:59"),
        ("thứ hai tuần sau",                {"date_type": "weekday", "weekday": 0, "week_offset": 1},   "2026-11-16 23:59"),
        ("thứ tư tuần sau hai giờ chiều",   {"date_type": "weekday", "weekday": 2, "week_offset": 1, "time": "14:00"}, "2026-11-18 14:00"),
        ("thứ bảy tuần sau",                {"date_type": "weekday", "weekday": 5, "week_offset": 1},   "2026-11-21 23:59"),
        ("ngày 25 tháng 11",                {"date_type": "calendar_date", "day_of_month": 25, "month": 11},  "2026-11-25 23:59"),
        ("ngày 5 tháng 1 (đã qua → năm sau)", {"date_type": "calendar_date", "day_of_month": 5, "month": 1}, "2027-01-05 23:59"),
        ("mua sách giáo trình",             {},                                 None),
        ("time bị hỏng",                    {"time": "25:99"},                  None),
        # small LLMs often write 0 instead of null: date_type must win
        ("lúc sáu giờ sáng, LLM điền 0",    {"time": "06:00", "day_offset": 0, "weekday": 0}, "2026-11-11 06:00"),
        ("không có deadline, LLM điền 0",   {"day_offset": 0, "weekday": 0, "day_of_month": 0, "month": 0}, None),
        ("thứ năm, day_offset rác",         {"date_type": "weekday", "weekday": 3, "day_offset": 0}, "2026-11-12 23:59"),
    ]
    for sentence, fields, expected in cases:
        result = resolve_deadline({**blank, **fields}, reference)
        got = result.strftime("%Y-%m-%d %H:%M") if result else None
        assert got == expected, f"{sentence!r}: got {got}, expected {expected}"
    print(f"✅ resolve_deadline: {len(cases)} cases pass")


check_resolve_deadline()


def build_relative_system_prompt(reference_time: str) -> str:
    """System prompt for the relative-time parser: the LLM reads time words, it never computes dates."""
    return (
        "Bạn trích xuất MỘT việc cần làm từ câu nói tiếng Việt của sinh viên.\n"
        f"Thời điểm hiện tại: {describe_reference_time(reference_time)}.\n"
        "- title: ngắn gọn, viết hoa chữ đầu, không chứa thời gian.\n"
        "- category: study (học/ôn), assignment (bài tập/nộp/báo cáo), exam (thi/kiểm tra), personal, other.\n"
        "- priority: high chỉ khi có 'gấp', 'quan trọng'; low nếu việc vặt; còn lại medium.\n"
        "- KHÔNG tự tính ngày. Chọn date_type trước, rồi chỉ điền trường của loại đó; các trường khác để null.\n"
        "- Câu chỉ có giờ (vd 'lúc sáu giờ sáng') thì date_type = none và chỉ điền time.\n"
        "- Câu không nhắc thời gian thì date_type = none và time = null."
    )


class RelativeTimeTodoParser(OllamaTodoParser):
    """LLM extracts relative time fields; Python computes deadline_at with resolve_deadline()."""

    def parse(self, transcript: str, reference_time: str) -> tuple[dict | None, float, str | None]:
        """Return (draft with deadline_at, seconds, error message or None)."""
        payload = {
            "model": self.model_name,
            "stream": False,
            "format": TODO_DRAFT_SCHEMA_RELATIVE,
            "options": {"temperature": 0},
            "messages": [{"role": "system", "content": build_relative_system_prompt(reference_time)},
                         {"role": "user", "content": transcript}],
        }
        if self.model_name.startswith("qwen3"):
            payload["think"] = False
        start = time.perf_counter()
        try:
            response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
            response.raise_for_status()
            draft = json.loads(response.json()["message"]["content"])
            deadline = resolve_deadline(draft, datetime.fromisoformat(reference_time))
            draft["deadline_at"] = deadline.isoformat() if deadline else None
            return draft, time.perf_counter() - start, None
        except (requests.RequestException, json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
            return None, time.perf_counter() - start, str(error)


# Results go into the same cache under the name "<model>+py", so the report shows them as extra rows.
PY_TEST_MODELS = ["qwen2.5:7b"]  # add "gemma3:4b" later if you want
for llm_model in PY_TEST_MODELS:
    cache_name = f"{llm_model}+py"
    done = {(r["source"], r["id"]) for r in load_jsonl_cache(PARSER_CACHE) if r["llm_model"] == cache_name}
    todo = [(s, e, t) for s, e, t in parser_inputs() if s == "gold" and (s, e["id"]) not in done]
    if not todo:
        print(f"{cache_name}: cached")
        continue
    parser = RelativeTimeTodoParser(llm_model)
    parser.parse("test", examples[0]["reference_time"])  # warm-up
    for source, example, transcript in todo:
        draft, seconds, error = parser.parse(transcript, example["reference_time"])
        append_jsonl(PARSER_CACHE, {"llm_model": cache_name, "source": source, "id": example["id"],
                                    "transcript": transcript, "draft": draft,
                                    "processing_seconds": round(seconds, 3), "error": error})

    print(f"{cache_name}: {len(todo)} done")

parser_results = load_jsonl_cache(PARSER_CACHE)

### 9b. Prompt variants (calendar table)

In [ ]:
# Prompt experiments: same LLM, same sentences, different system prompt → compare against the original prompt
from datetime import datetime, timedelta

PROMPT_TEST_MODELS = ["qwen2.5:7b", "gemma3:4b"]  # the two best LLMs so far
DAY_LABELS = {0: "hôm nay", 1: "mai", 2: "mốt"}
CALENDAR_DAYS = 21


def build_calendar_table(reference_time: str, days: int = CALENDAR_DAYS) -> str:
    """List the next `days` dates with weekday, '+N ngày', 'tuần này/tuần sau/tuần sau nữa' and 'hôm nay/mai/mốt'."""
    reference = datetime.fromisoformat(reference_time)
    reference_week = reference.isocalendar()[1]
    week_labels = {0: "tuần này", 1: "tuần sau", 2: "tuần sau nữa"}
    lines = []
    for offset in range(days):
        day = reference + timedelta(days=offset)
        week_label = week_labels.get(day.isocalendar()[1] - reference_week, "")
        extra = f", {DAY_LABELS[offset]}" if offset in DAY_LABELS else ""
        lines.append(f"- {VIETNAMESE_WEEKDAYS[day.weekday()]} {day:%Y-%m-%d} (+{offset} ngày, {week_label}{extra})")
    return "\n".join(lines)


def calendar_rules(reference_time: str) -> str:
    """Shared rules + calendar table used by every calendar-based variant."""
    return (
        "Bạn trích xuất MỘT việc cần làm từ câu nói tiếng Việt của sinh viên.\n"
        f"Thời điểm hiện tại: {describe_reference_time(reference_time)}.\n"
        f"Lịch {CALENDAR_DAYS} ngày tới (TRA BẢNG NÀY, không tự tính ngày; 'N ngày nữa' = dòng +N ngày):\n"
        f"{build_calendar_table(reference_time)}\n"
        "Quy tắc:\n"
        "- title: ngắn gọn, viết hoa chữ đầu, không chứa thời gian.\n"
        "- category: study (học/ôn), assignment (bài tập/nộp/báo cáo), exam (thi/kiểm tra), personal, other.\n"
        "- priority: high chỉ khi có 'gấp', 'quan trọng'; low nếu việc vặt; còn lại medium.\n"
        "- deadline_at: đúng định dạng YYYY-MM-DDTHH:MM:00+07:00.\n"
        "- 'thứ X' không kèm 'tuần sau' = thứ X gần nhất trong bảng tính từ hôm nay. 'cuối tuần' = Chủ Nhật.\n"
        "- Có ngày mà không có giờ: dùng 23:59. Có giờ mà không có ngày: hôm nay nếu giờ đó còn sau "
        "thời điểm hiện tại, nếu không thì ngày mai.\n"
        "- Câu KHÔNG nhắc ngày hay giờ nào: deadline_at = null. Tuyệt đối không tự đặt deadline."
    )


def prompt_calendar(reference_time: str) -> str:
    """Variant 1: calendar table, the LLM looks dates up instead of computing them."""
    return calendar_rules(reference_time)


def prompt_calendar_fewshot(reference_time: str) -> str:
    """Variant 2: calendar table + worked examples (sentences NOT in the test set), dates computed for this reference."""
    reference = datetime.fromisoformat(reference_time)
    monday = reference - timedelta(days=reference.weekday())

    def this_or_next(weekday: int) -> datetime:
        """Nearest given weekday from today on (same rule as the eval labels)."""
        day = monday + timedelta(days=weekday)
        return day if day.date() >= reference.date() else day + timedelta(days=7)

    nine_pm_today = reference.replace(hour=21, minute=0)
    nine_pm = nine_pm_today if nine_pm_today > reference else nine_pm_today + timedelta(days=1)
    examples = [
        ("thứ bảy này đi siêu thị", "Đi siêu thị", "personal", "low", this_or_next(5).replace(hour=23, minute=59)),
        ("tuần sau thứ sáu nộp bài thu hoạch", "Nộp bài thu hoạch", "assignment", "medium",
         (monday + timedelta(days=7 + 4)).replace(hour=23, minute=59)),
        ("chín giờ tối gọi cho bạn", "Gọi cho bạn", "personal", "medium", nine_pm),
        ("mốt thi vấn đáp tiếng Anh", "Thi vấn đáp tiếng Anh", "exam", "medium",
         (reference + timedelta(days=2)).replace(hour=23, minute=59)),
        ("nhớ trả sách thư viện", "Trả sách thư viện", "personal", "low", None),
    ]
    shots = "\n".join(
        f'"{text}" → {json.dumps({"title": title, "category": category, "priority": priority, "deadline_at": deadline.strftime("%Y-%m-%dT%H:%M:00+07:00") if deadline else None}, ensure_ascii=False)}'
        for text, title, category, priority, deadline in examples)
    return calendar_rules(reference_time) + "\nVí dụ:\n" + shots


# --- Category rules v2: no inline examples (small LLMs copy them into the title), strict "ôn" rule ---
CATEGORY_RULES_V2 = (
    "- category, xét theo thứ tự, khớp cái nào trước thì dừng:\n"
    "  1. exam: CHỈ KHI câu có tên một bài thi: 'thi', 'kiểm tra', 'quiz', 'midterm', 'cuối kỳ', 'giữa kỳ', 'vấn đáp'.\n"
    "     'ôn' mà không có các từ đó thì KHÔNG phải exam.\n"
    "  2. assignment: làm ra và nộp một sản phẩm học tập: bài tập, báo cáo, đồ án, tiểu luận, lab, slide, code, project;\n"
    "     kể cả các bước làm ra nó (họp nhóm đồ án, tìm tài liệu cho tiểu luận, fix bug project, push code).\n"
    "  3. study: học, ôn, đọc, đi thư viện, học nhóm để hiểu bài; không gắn với bài thi hay sản phẩm nộp.\n"
    "  4. personal: đời sống cá nhân: sức khoẻ, thể thao, gia đình, nhà cửa, mua sắm, giải trí, hồ sơ xin việc.\n"
    "  5. other: thủ tục ở trường (nộp đơn, đăng ký học phần, đóng học phí, gặp giảng viên, hỏi điểm, câu lạc bộ),\n"
    "     giúp người khác, họp không rõ việc gì, và mọi việc còn lại.\n"
)


def prompt_calendar_and_category_v2(reference_time: str) -> str:
    """Variant 5: calendar table (best deadline prompt) + category rules v2, no examples."""
    rules = calendar_rules(reference_time)
    old_line = "- category: study (học/ôn), assignment (bài tập/nộp/báo cáo), exam (thi/kiểm tra), personal, other.\n"
    assert old_line in rules
    return rules.replace(old_line, CATEGORY_RULES_V2)


PROMPT_TEST_SOURCES = ["gold", "asr:vinai/PhoWhisper-medium"]   # end-to-end on the ASR we plan to deploy
PROMPT_VARIANTS = {  # name → function(reference_time) -> system prompt. Add your own here.
    "cal21": prompt_calendar,
    "cal21_cat2": prompt_calendar_and_category_v2,
}


class PromptVariantTodoParser(OllamaTodoParser):
    """Same request as OllamaTodoParser (same schema), only the system prompt is swapped."""

    def __init__(self, model_name: str, build_prompt):
        """Remember the model and the function that builds the system prompt."""
        super().__init__(model_name)
        self.build_prompt = build_prompt

    def parse(self, transcript: str, reference_time: str) -> tuple[dict | None, float, str | None]:
        """Return (draft or None, seconds, error message or None)."""
        payload = {
            "model": self.model_name,
            "stream": False,
            "format": TODO_DRAFT_SCHEMA,
            "options": {"temperature": 0},
            "messages": [{"role": "system", "content": self.build_prompt(reference_time)},
                         {"role": "user", "content": transcript}],
        }
        if self.model_name.startswith("qwen3"):
            payload["think"] = False
        start = time.perf_counter()
        try:
            response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
            response.raise_for_status()
            draft = json.loads(response.json()["message"]["content"])
            return draft, time.perf_counter() - start, None
        except (requests.RequestException, json.JSONDecodeError, KeyError) as error:
            return None, time.perf_counter() - start, str(error)


preview = prompt_calendar(examples[0]["reference_time"])
assert "+14 ngày" in preview, "old 14-day table still in use"
print("Prompt preview (cal21):\n" + preview + "\n")

# --- run every (model, variant); cached as llm_model = "<model>|<variant>" ---
for llm_model in PROMPT_TEST_MODELS:
    for variant_name, build_prompt in PROMPT_VARIANTS.items():
        cache_name = f"{llm_model}|{variant_name}"
        done = {(r["source"], r["id"]) for r in load_jsonl_cache(PARSER_CACHE) if r["llm_model"] == cache_name}
        todo = [(s, e, t) for s, e, t in parser_inputs() if s in PROMPT_TEST_SOURCES and (s, e["id"]) not in done]
        if not todo:
            print(f"{cache_name}: cached")
            continue
        parser = PromptVariantTodoParser(llm_model, build_prompt)
        for source, example, transcript in todo:
            draft, seconds, error = parser.parse(transcript, example["reference_time"])
            append_jsonl(PARSER_CACHE, {"llm_model": cache_name, "source": source, "id": example["id"],
                                        "transcript": transcript, "draft": draft,
                                        "processing_seconds": round(seconds, 3), "error": error})
        print(f"{cache_name}: {len(todo)} done")

parser_results = load_jsonl_cache(PARSER_CACHE)

# --- compare each variant with the original prompt of the same model (paired, McNemar) ---
variant_rows = []
for llm_model in PROMPT_TEST_MODELS:
    for source in PROMPT_TEST_SOURCES:
        baseline = task_success_by_id(parser_results, llm_model, source)
        for variant_name in ["(original)"] + list(PROMPT_VARIANTS):
            run_name = llm_model if variant_name == "(original)" else f"{llm_model}|{variant_name}"
            rows = [r for r in parser_results if r["llm_model"] == run_name and r["source"] == source]
            if not rows:
                continue
            drafts = [r["draft"] or {} for r in rows]
            expected = [by_id[r["id"]]["expected"] for r in rows]
            row = {"llm": llm_model, "prompt": variant_name, "n": len(rows),
                   "category_acc": round(accuracy([d.get("category") for d in drafts], [e["category"] for e in expected]), 3),
                   "deadline_day_acc": round(sum(same_deadline_day(d.get("deadline_at"), e["deadline_at"])
                                                 for d, e in zip(drafts, expected)) / len(rows), 3),
                   "invented_deadline": sum(1 for d, e in zip(drafts, expected)
                                            if e["deadline_at"] is None and d.get("deadline_at")),
                   "deadline_invalid": sum(is_invalid_deadline(d.get("deadline_at")) for d in drafts),
                   "TASK_SUCCESS": round(sum(is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows) / len(rows), 3),
                   "latency_s": round(sum(r["processing_seconds"] for r in rows) / len(rows), 2)}
            if variant_name != "(original)":
                paired = PairedComparison("original", baseline, variant_name,
                                          task_success_by_id(parser_results, run_name, source)).summary()
                row.update({"only_orig_ok": paired["only_A_ok"], "only_new_ok": paired["only_B_ok"],
                            "p_value": paired["p_value"], "verdict": paired["verdict"]})
            variant_rows.append(row)

print("Prompt variants vs original prompt   (invented_deadline = deadline given to a sentence that has none)")
display(pd.DataFrame(variant_rows))

### 9c. Head-to-head

In [ ]:
# Head-to-head: LLM looks dates up in a table (calendar) vs Python computes them (+py)
for run_a, run_b in [("qwen2.5:7b|calendar", "qwen2.5:7b+py"), ("qwen2.5:7b|cal21", "qwen2.5:7b+py"),
                     ("qwen2.5:7b|calendar", "qwen2.5:7b|cal21")]:
    success_a = task_success_by_id(parser_results, run_a, "gold")
    success_b = task_success_by_id(parser_results, run_b, "gold")
    if success_a and success_b:
        print(PairedComparison(run_a, success_a, run_b, success_b).summary())
    else:
        print(f"skip {run_a} vs {run_b}: one of them has not run yet")

### 9d. Remaining wrong deadlines

In [ ]:
# Which run to inspect: "qwen2.5:7b|calendar", "qwen2.5:7b|cal21" or "qwen2.5:7b+py"
STILL_WRONG_RUN = "qwen2.5:7b+py"
still_wrong = [{"id": r["id"], "tags": ", ".join(t for t in by_id[r["id"]]["tags"] if t != "audio"),
                "text": r["transcript"][:50],
                "expected": by_id[r["id"]]["expected"]["deadline_at"], "predicted": (r["draft"] or {}).get("deadline_at")}
               for r in parser_results
               if r["llm_model"] == STILL_WRONG_RUN and r["source"] == "gold"
               and not same_deadline_day((r["draft"] or {}).get("deadline_at"), by_id[r["id"]]["expected"]["deadline_at"])]
print(STILL_WRONG_RUN, ":", len(still_wrong), "wrong deadlines")
display(pd.DataFrame(still_wrong))

### 9f. Rule-based deadline parser

The LLM keeps title / category / priority; `deadline_at` comes from `time_parser.py` (pure rules, ~30 µs). Nothing is re-queried: cached drafts are re-scored.

In [ ]:
# === server/app/service/voice/time_parser.py (copied verbatim, keep in sync) ===
import re
import unicodedata
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from datetime import time as clock_time

VIETNAM_TIMEZONE = timezone(timedelta(hours=7))
END_OF_DAY = clock_time(23, 59)

UNIT_WORDS = {"không": 0, "một": 1, "hai": 2, "ba": 3, "bốn": 4, "năm": 5, "sáu": 6, "bảy": 7, "tám": 8, "chín": 9}
UNIT_WORDS_AFTER_TENS = {**UNIT_WORDS, "mốt": 1, "tư": 4, "lăm": 5, "nhăm": 5}  # "hai mươi mốt", "mười lăm"
TENS_MARKERS = {"mươi", "chục"}
WEEKDAY_WORDS = {"hai": 0, "ba": 1, "tư": 2, "bốn": 2, "năm": 3, "sáu": 4, "bảy": 5}  # after "thứ"; Monday = 0
PARTS_OF_DAY = {"sáng", "trưa", "chiều", "tối", "đêm", "khuya"}
RELATIVE_DAY_WORDS = {"nay": 0, "mai": 1, "mốt": 2}


@dataclass
class TimeExpression:
    """Time information found in one sentence, before it is turned into a concrete datetime."""

    date_type: str = "none"          # none | relative_days | weekday | calendar_date
    day_offset: int | None = None
    weekday: int | None = None       # Monday = 0 ... Sunday = 6
    week_offset: int = 0             # 1 = "tuần sau", 2 = "tuần sau nữa"
    day_of_month: int | None = None
    month: int | None = None
    hour: int | None = None          # 24h, after applying the part of day
    minute: int = 0
    matched: list[str] = field(default_factory=list)  # phrases that triggered each rule (for debugging)

    def has_clock(self) -> bool:
        """True when a clock time was found."""
        return self.hour is not None


def normalize_sentence(text: str) -> list[str]:
    """NFC-normalize, lowercase, split '8h30' / '19:30' into words and return the tokens without punctuation."""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"(\d{1,2})\s*[h:]\s*(\d{2})\b", r"\1 giờ \2", text)  # 8h30, 19:30 → 8 giờ 30
    text = re.sub(r"(\d{1,2})\s*h\b", r"\1 giờ", text)                  # 19h → 19 giờ
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()


def parse_vietnamese_number(tokens: list[str]) -> int | None:
    """Spoken Vietnamese number made of exactly these tokens (0..99), or None.

    "bảy" = 7 · "mười" = 10 · "mười lăm" = 15 · "hai mươi" = 20 · "hai mươi mốt" = 21 · "19" = 19
    value = tens · 10 + units   ("mười" alone means tens = 1)
    """
    if not tokens:
        return None
    if len(tokens) == 1 and tokens[0].isdigit():
        return int(tokens[0])
    if len(tokens) == 1:
        return UNIT_WORDS.get(tokens[0]) if tokens[0] != "mười" else 10
    if tokens[0] == "mười":
        tens, rest = 1, tokens[1:]
    elif tokens[0] in UNIT_WORDS and tokens[1] in TENS_MARKERS:
        tens, rest = UNIT_WORDS[tokens[0]], tokens[2:]
    else:
        return None
    if not rest:
        return tens * 10
    if len(rest) == 1 and rest[0] in UNIT_WORDS_AFTER_TENS:
        unit = UNIT_WORDS_AFTER_TENS[rest[0]]
        if rest[0] == "mốt" and tens < 2:  # "mười mốt" is not a number, "mười một" is
            return None
        return tens * 10 + unit
    return None


def longest_number_ending_at(tokens: list[str], end: int, max_length: int = 3) -> tuple[int, int] | None:
    """Longest number whose last token is tokens[end - 1]; returns (value, start index) or None."""
    for length in range(max_length, 0, -1):
        start = end - length
        if start >= 0:
            value = parse_vietnamese_number(tokens[start:end])
            if value is not None:
                return value, start
    return None


def longest_number_starting_at(tokens: list[str], start: int, max_length: int = 3) -> tuple[int, int] | None:
    """Longest number whose first token is tokens[start]; returns (value, end index) or None."""
    for length in range(max_length, 0, -1):
        end = start + length
        if end <= len(tokens):
            value = parse_vietnamese_number(tokens[start:end])
            if value is not None:
                return value, end
    return None


def to_24_hour(hour: int, part_of_day: str | None) -> int:
    """Apply the spoken part of day to a 12h clock hour.

    chiều/tối: h < 12 → h + 12 · trưa: h ≤ 4 → h + 12 (một giờ trưa = 13) · đêm/khuya: 6 ≤ h < 12 → h + 12
    sáng or no part of day: unchanged
    """
    if hour >= 13 or part_of_day is None or part_of_day == "sáng":
        return hour
    if part_of_day in ("chiều", "tối"):
        return hour + 12 if hour < 12 else hour
    if part_of_day == "trưa":
        return hour + 12 if hour <= 4 else hour
    if part_of_day in ("đêm", "khuya"):
        return hour + 12 if 6 <= hour < 12 else hour
    return hour


def date_from_weekday(reference_day: date, weekday: int, week_offset: int) -> date:
    """Named weekday, optionally in a later week (Monday = 0 ... Sunday = 6).

    monday = reference_date − reference_date.weekday()
    date   = monday + 7·week_offset + weekday
    if week_offset = 0 and date < reference_date: date += 7   (that day already passed this week)
    """
    monday = reference_day - timedelta(days=reference_day.weekday())
    target = monday + timedelta(days=7 * week_offset + weekday)
    if week_offset == 0 and target < reference_day:
        target += timedelta(days=7)
    return target


def date_from_day_and_month(reference_day: date, day_of_month: int, month: int | None) -> date | None:
    """Explicit 'ngày N (tháng M)': the next such date on or after today; None if it does not exist."""
    try:
        if month is None:
            target = reference_day.replace(day=day_of_month)
            if target < reference_day:
                next_month_first = (reference_day.replace(day=1) + timedelta(days=32)).replace(day=1)
                target = next_month_first.replace(day=day_of_month)
            return target
        target = date(reference_day.year, month, day_of_month)
        return target if target >= reference_day else date(reference_day.year + 1, month, day_of_month)
    except ValueError:  # 31/2, day 31 in a 30-day month, ...
        return None


class VietnameseTimeParser:
    """Finds time expressions in a Vietnamese sentence and turns them into a deadline."""

    def parse(self, text: str) -> TimeExpression:
        """Extract date and clock information from one sentence (no reference time needed yet)."""
        tokens = normalize_sentence(text)
        expression = TimeExpression()
        self._find_clock(tokens, expression)
        if not self._find_calendar_date(tokens, expression):
            if not self._find_relative_days(tokens, expression):
                self._find_weekday(tokens, expression)
        return expression

    def resolve(self, expression: TimeExpression, reference: datetime) -> datetime | None:
        """Turn a TimeExpression into an absolute deadline (+07:00), or None when no time was said.

        calendar_date → that date · relative_days → today + day_offset · weekday → see date_from_weekday
        clock only    → today if that time is still ahead, otherwise tomorrow
        clock: the spoken time, or 23:59 when only a day was said
        """
        reference = reference if reference.tzinfo else reference.replace(tzinfo=VIETNAM_TIMEZONE)
        reference = reference.astimezone(VIETNAM_TIMEZONE)
        today = reference.date()
        clock = clock_time(expression.hour, expression.minute) if expression.has_clock() else None

        if expression.date_type == "calendar_date":
            target_day = date_from_day_and_month(today, expression.day_of_month, expression.month)
        elif expression.date_type == "relative_days":
            target_day = today + timedelta(days=expression.day_offset)
        elif expression.date_type == "weekday":
            target_day = date_from_weekday(today, expression.weekday, expression.week_offset)
        elif clock is not None:
            today_at_clock = datetime.combine(today, clock, VIETNAM_TIMEZONE)
            target_day = today if today_at_clock > reference else today + timedelta(days=1)
        else:
            return None

        if target_day is None:
            return None
        return datetime.combine(target_day, clock or END_OF_DAY, VIETNAM_TIMEZONE)

    def extract_deadline(self, text: str, reference: datetime) -> datetime | None:
        """parse() + resolve() in one call."""
        return self.resolve(self.parse(text), reference)

    # --- rules -------------------------------------------------------------------------------------

    def _find_clock(self, tokens: list[str], expression: TimeExpression) -> None:
        """'<number> giờ [rưỡi | <number> (phút) | kém <number> (phút)]' + nearest part of day."""
        for index, token in enumerate(tokens):
            if token != "giờ":
                continue
            hour_match = longest_number_ending_at(tokens, index)
            if hour_match is None or not 0 <= hour_match[0] <= 24:
                continue  # "bây giờ", "giờ học", ...
            hour, minute, after = hour_match[0], 0, index + 1
            if after < len(tokens) and tokens[after] == "rưỡi":
                minute, after = 30, after + 1
            elif after < len(tokens) and tokens[after] == "kém":
                minutes_before = longest_number_starting_at(tokens, after + 1, max_length=2)
                if minutes_before and 0 < minutes_before[0] < 60:
                    hour, minute, after = (hour - 1) % 24, 60 - minutes_before[0], minutes_before[1]
            else:
                minutes_match = longest_number_starting_at(tokens, after, max_length=3)
                if minutes_match and 0 <= minutes_match[0] < 60:
                    minute, after = minutes_match[0], minutes_match[1]
            if after < len(tokens) and tokens[after] == "phút":
                after += 1
            expression.hour = to_24_hour(hour, self._nearest_part_of_day(tokens, index))
            expression.minute = minute
            expression.matched.append(" ".join(tokens[hour_match[1]:after]))
            return
        self._find_half_hour_without_gio(tokens, expression)

    def _find_half_hour_without_gio(self, tokens: list[str], expression: TimeExpression) -> None:
        """'(lúc) <number> rưỡi <part of day>' without the word 'giờ', e.g. 'lúc ba rưỡi chiều' = 15:30.

        Needs 'lúc' before or a part of day right after, so 'một năm rưỡi' (1.5 years) is not a clock.
        """
        for index, token in enumerate(tokens):
            if token != "rưỡi":
                continue
            hour_match = longest_number_ending_at(tokens, index, max_length=2)
            if hour_match is None or not 1 <= hour_match[0] <= 12:
                continue
            before = tokens[hour_match[1] - 1] if hour_match[1] > 0 else ""
            after = tokens[index + 1] if index + 1 < len(tokens) else ""
            if before != "lúc" and after not in PARTS_OF_DAY:
                continue
            expression.hour = to_24_hour(hour_match[0], self._nearest_part_of_day(tokens, index))
            expression.minute = 30
            expression.matched.append(" ".join(tokens[hour_match[1]:index + 1]))
            return

    @staticmethod
    def _nearest_part_of_day(tokens: list[str], clock_index: int) -> str | None:
        """The part-of-day word (sáng/trưa/chiều/tối/đêm) closest to the clock, preferring the one after it."""
        positions = [i for i, token in enumerate(tokens) if token in PARTS_OF_DAY]
        if not positions:
            return None
        closest = min(positions, key=lambda i: (abs(i - clock_index), i < clock_index))
        return tokens[closest]

    def _find_calendar_date(self, tokens: list[str], expression: TimeExpression) -> bool:
        """'ngày|mùng|mồng <number> [tháng <number>]' → explicit date."""
        for index, token in enumerate(tokens):
            if token not in ("ngày", "mùng", "mồng") or index + 1 >= len(tokens):
                continue
            if tokens[index + 1] in RELATIVE_DAY_WORDS or tokens[index + 1] in ("kia", "nữa"):
                continue  # "ngày mai", "ngày mốt", "ngày kia"
            day_match = longest_number_starting_at(tokens, index + 1)
            if day_match is None or not 1 <= day_match[0] <= 31:
                continue
            month = None
            if day_match[1] < len(tokens) and tokens[day_match[1]] == "tháng":
                month_match = longest_number_starting_at(tokens, day_match[1] + 1, max_length=2)
                if month_match and 1 <= month_match[0] <= 12:
                    month = month_match[0]
            expression.date_type, expression.day_of_month, expression.month = "calendar_date", day_match[0], month
            expression.matched.append(" ".join(tokens[index:day_match[1] + (2 if month else 0)]))
            return True
        return False

    def _find_relative_days(self, tokens: list[str], expression: TimeExpression) -> bool:
        """'<number> ngày|tuần nữa/tới', 'hôm nay / tối nay', 'mai', 'mốt', 'ngày kia' → offset in days."""
        for index, token in enumerate(tokens):
            if token in ("ngày", "tuần") and index + 1 < len(tokens) and tokens[index + 1] in ("nữa", "tới"):
                count = longest_number_ending_at(tokens, index)
                if count is not None:
                    days = count[0] * (7 if token == "tuần" else 1)
                    self._set_relative(expression, days, " ".join(tokens[count[1]:index + 2]))
                    return True
        for index, token in enumerate(tokens):
            previous = tokens[index - 1] if index > 0 else ""
            if token == "kia" and previous == "ngày":
                self._set_relative(expression, 2, "ngày kia")
                return True
            if token in RELATIVE_DAY_WORDS and previous not in TENS_MARKERS:  # not "hai mươi mốt"
                self._set_relative(expression, RELATIVE_DAY_WORDS[token], f"{previous} {token}".strip())
                return True
        return False

    @staticmethod
    def _set_relative(expression: TimeExpression, days: int, phrase: str) -> None:
        """Store a relative-days result."""
        expression.date_type, expression.day_offset = "relative_days", days
        expression.matched.append(phrase)

    def _find_weekday(self, tokens: list[str], expression: TimeExpression) -> bool:
        """'thứ hai..thứ bảy', 'thứ 2..7', 'chủ nhật', 'cuối tuần' + optional 'tuần sau/tới (nữa)'."""
        weekday, phrase = None, ""
        for index, token in enumerate(tokens[:-1]):
            following = tokens[index + 1]
            if token == "thứ" and following in WEEKDAY_WORDS:
                weekday, phrase = WEEKDAY_WORDS[following], f"thứ {following}"
            elif token == "thứ" and following.isdigit() and 2 <= int(following) <= 7:
                weekday, phrase = int(following) - 2, f"thứ {following}"
            elif (token, following) == ("chủ", "nhật"):
                weekday, phrase = 6, "chủ nhật"
            elif (token, following) == ("cuối", "tuần"):
                weekday, phrase = 6, "cuối tuần"
            if weekday is not None:
                break
        if weekday is None:
            return False
        expression.date_type, expression.weekday = "weekday", weekday
        expression.week_offset = self._week_offset(tokens)
        expression.matched.append(phrase)
        return True

    @staticmethod
    def _week_offset(tokens: list[str]) -> int:
        """1 for 'tuần sau' / 'tuần tới', 2 for 'tuần sau nữa', else 0."""
        text = " ".join(tokens)
        if re.search(r"\btuần (sau|tới) nữa\b", text):
            return 2
        if re.search(r"\btuần (sau|tới)\b", text):
            return 1
        return 0

In [ ]:
# Rule-based deadline (server/app/service/voice/time_parser.py) replaces the LLM's deadline_at.
# No LLM calls: every cached draft is re-scored with  deadline_at = VietnameseTimeParser().extract_deadline(transcript).
rule_parser = VietnameseTimeParser()


def rule_deadline(transcript: str, reference_time: str) -> str | None:
    """Deadline from the rule parser as an ISO string, or None when the sentence has no time expression."""
    deadline = rule_parser.extract_deadline(transcript, datetime.fromisoformat(reference_time))
    return deadline.isoformat() if deadline else None


def draft_with_rule_deadline(row: dict) -> dict | None:
    """The cached LLM draft (title/category/priority) with its deadline replaced by the rule parser's."""
    if not row["draft"]:
        return None
    return {**row["draft"], "deadline_at": rule_deadline(row["transcript"], by_id[row["id"]]["reference_time"])}


def rule_success_by_id(results: list[dict], llm_model: str, source: str) -> dict[str, bool]:
    """id → task success when the rule parser supplies the deadline."""
    return {r["id"]: is_task_success(draft_with_rule_deadline(r), by_id[r["id"]]["expected"])
            for r in results if r["llm_model"] == llm_model and r["source"] == source}


# --- rule parser alone: deadline accuracy per transcript source ---
rule_rows = []
for source, group in pd.DataFrame(parser_results).drop_duplicates(["source", "id"]).groupby("source"):
    rows = group.to_dict("records")
    ok = [same_deadline_day(rule_deadline(r["transcript"], by_id[r["id"]]["reference_time"]),
                            by_id[r["id"]]["expected"]["deadline_at"]) for r in rows]
    rule_rows.append({"source": source, "n": len(rows), "rule deadline_day_acc": round(sum(ok) / len(ok), 3)})
print("Rule parser alone")
display(pd.DataFrame(rule_rows))

# --- every (LLM run, source): TSR with the LLM's own deadline vs with the rule deadline ---
compare_rows = []
for (llm_model, source), group in pd.DataFrame(parser_results).groupby(["llm_model", "source"]):
    llm_ok = task_success_by_id(parser_results, llm_model, source)
    rule_ok = rule_success_by_id(parser_results, llm_model, source)
    paired = PairedComparison("llm", llm_ok, "rule", rule_ok).summary()
    drafts = [r["draft"] or {} for r in group.to_dict("records")]
    expected = [by_id[i]["expected"] for i in group["id"]]
    compare_rows.append({"llm_model": llm_model, "source": source, "n": paired["n"],
                         "category_acc": round(accuracy([d.get("category") for d in drafts], [e["category"] for e in expected]), 3),
                         "TSR llm-deadline": paired["TSR_A"], "TSR rule-deadline": paired["TSR_B"],
                         "only_llm_ok": paired["only_A_ok"], "only_rule_ok": paired["only_B_ok"], "p_value": paired["p_value"]})
compare_table = pd.DataFrame(compare_rows).sort_values("TSR rule-deadline", ascending=False)
print("TSR: LLM deadline vs rule deadline (sorted by rule-deadline TSR)")
display(compare_table)

# --- where the rule parser still fails (usually ASR misheard the time words) ---
rule_failures = [{"source": r["source"], "id": r["id"], "transcript": r["transcript"][:60],
                  "expected": by_id[r["id"]]["expected"]["deadline_at"],
                  "rule": rule_deadline(r["transcript"], by_id[r["id"]]["reference_time"]),
                  "matched": rule_parser.parse(r["transcript"]).matched}
                 for r in pd.DataFrame(parser_results).drop_duplicates(["source", "id"]).to_dict("records")
                 if not same_deadline_day(rule_deadline(r["transcript"], by_id[r["id"]]["reference_time"]),
                                          by_id[r["id"]]["expected"]["deadline_at"])]
print(f"{len(rule_failures)} rule-parser deadline failures")
display(pd.DataFrame(rule_failures))

### 9g. A2a: LLM repairs ASR mishearings before extraction

`corrected_transcript` is the first schema field. Runs qwen2.5:7b on gold + ASR medium (~5 min). Needs 9b (prompt_calendar_and_category_v2) and 9f (rule_deadline) to have run.

In [ ]:
# A2a: let the LLM repair ASR mishearings BEFORE extracting the todo.
# Schema gets `corrected_transcript` as the FIRST field, so the model "thinks" about the sentence before the title.
# Deadline is still computed by the rule parser, now on corrected_transcript (does repair help "hứa sáo" → "thứ sáu"?).

A2A_MODEL = "qwen2.5:7b"
A2A_SOURCES = ["gold", "asr:vinai/PhoWhisper-medium"]   # gold = must not get worse; medium = the real test

# Subjects the app knows (put the real course list here; NOT words copied from the eval set's transcripts)
KNOWN_SUBJECTS = ["Giải tích", "Đại số tuyến tính", "Xác suất thống kê", "Toán rời rạc", "Vật lý đại cương",
                  "Lập trình hướng đối tượng", "Cấu trúc dữ liệu và giải thuật", "Cơ sở dữ liệu", "Mạng máy tính",
                  "Hệ điều hành", "Kiến trúc máy tính", "Trí tuệ nhân tạo", "Học máy", "Phân tích thiết kế hệ thống",
                  "Tư tưởng Hồ Chí Minh", "Triết học Mác Lê nin", "Kinh tế chính trị", "Tiếng Anh chuyên ngành"]

TODO_DRAFT_SCHEMA_A2A = {
    "type": "object",
    "properties": {
        "corrected_transcript": {"type": "string",
                                 "description": "Câu gốc sau khi sửa lỗi nghe; giữ nguyên mọi số, giờ, ngày, thứ"},
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "assignment", "exam", "personal", "other"]},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "deadline_at": {"type": ["string", "null"], "description": "ISO 8601 with +07:00 offset, or null"},
    },
    "required": ["corrected_transcript", "title", "category", "priority", "deadline_at"],
}

ASR_REPAIR_RULES = (
    "Câu đưa vào là kết quả nhận dạng giọng nói, có thể sai ở những từ NGHE GIỐNG NHAU (ví dụ 'đặc lịch mời' là "
    "'đặt lịch mới'), và các từ tiếng Anh chuyên ngành IT (deadline, GitHub, Python, OOP, quiz, midterm, lab, slide, "
    "project, bug, commit) thường bị ghi thành tiếng Việt vô nghĩa.\n"
    "- Bước 1, corrected_transcript: suy ra câu người nói thật sự nói. Chỉ sửa từ nghe nhầm; KHÔNG đổi số, giờ, "
    "ngày, thứ; không thêm bớt ý.\n"
    f"- Tên môn học có thể gặp: {', '.join(KNOWN_SUBJECTS)}.\n"
    "- Bước 2: trích xuất title / category / priority / deadline_at TỪ corrected_transcript.\n"
)

REPAIR_EXAMPLES = (  # error TYPES seen in real ASR output, but these sentences are NOT in the eval set
    "Ví dụ sửa lỗi nghe (chỉ minh hoạ cách sửa):\n"
    "  'đặc lịch mời đi khám răng lúc hai giờ' → 'đặt lịch mới đi khám răng lúc hai giờ'\n"
    "  'hứa ba họp câu lạc bộ' → 'thứ ba họp câu lạc bộ'\n"
    "  'nộp bài láp môn cơ sở dữ liệu' → 'nộp bài lab môn cơ sở dữ liệu'\n"
)


def prompt_a2a(reference_time: str) -> str:
    """A2a: calendar + category v2 prompt, plus ASR-repair instructions (no examples)."""
    return ASR_REPAIR_RULES + prompt_calendar_and_category_v2(reference_time)


def prompt_a2a_fewshot(reference_time: str) -> str:
    """A2a + three repair examples (watch for leakage into titles, as happened with gemma)."""
    return ASR_REPAIR_RULES + REPAIR_EXAMPLES + prompt_calendar_and_category_v2(reference_time)


class SchemaPromptTodoParser(PromptVariantTodoParser):
    """PromptVariantTodoParser with a custom JSON schema (needed for the extra corrected_transcript field)."""

    def __init__(self, model_name: str, build_prompt, schema: dict):
        """Remember model, prompt builder and schema."""
        super().__init__(model_name, build_prompt)
        self.schema = schema

    def parse(self, transcript: str, reference_time: str) -> tuple[dict | None, float, str | None]:
        """Return (draft or None, seconds, error message or None)."""
        payload = {
            "model": self.model_name, "stream": False, "format": self.schema, "options": {"temperature": 0},
            "messages": [{"role": "system", "content": self.build_prompt(reference_time)},
                         {"role": "user", "content": transcript}],
        }
        start = time.perf_counter()
        try:
            response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
            response.raise_for_status()
            return json.loads(response.json()["message"]["content"]), time.perf_counter() - start, None
        except (requests.RequestException, json.JSONDecodeError, KeyError) as error:
            return None, time.perf_counter() - start, str(error)


A2A_VARIANTS = {"a2a": prompt_a2a, "a2a_fewshot": prompt_a2a_fewshot}

for variant_name, build_prompt in A2A_VARIANTS.items():
    cache_name = f"{A2A_MODEL}|{variant_name}"
    done = {(r["source"], r["id"]) for r in load_jsonl_cache(PARSER_CACHE) if r["llm_model"] == cache_name}
    todo = [(s, e, t) for s, e, t in parser_inputs() if s in A2A_SOURCES and (s, e["id"]) not in done]
    if not todo:
        print(f"{cache_name}: cached")
        continue
    parser = SchemaPromptTodoParser(A2A_MODEL, build_prompt, TODO_DRAFT_SCHEMA_A2A)
    for source, example, transcript in todo:
        draft, seconds, error = parser.parse(transcript, example["reference_time"])
        append_jsonl(PARSER_CACHE, {"llm_model": cache_name, "source": source, "id": example["id"],
                                    "transcript": transcript, "draft": draft,
                                    "processing_seconds": round(seconds, 3), "error": error})
    print(f"{cache_name}: {len(todo)} done")

parser_results = load_jsonl_cache(PARSER_CACHE)


# --- scoring: baseline (cal21_cat2) vs a2a variants, rule deadline on raw vs corrected transcript ---
def corrected_text(row: dict) -> str:
    """The LLM-repaired transcript when present, else the raw one."""
    return (row["draft"] or {}).get("corrected_transcript") or row["transcript"]


def success_with_rule_deadline(row: dict, text_for_deadline: str) -> bool:
    """Task success using the cached draft's title/category and the rule deadline from `text_for_deadline`."""
    if not row["draft"]:
        return False
    example = by_id[row["id"]]
    draft = {**row["draft"], "deadline_at": rule_deadline(text_for_deadline, example["reference_time"])}
    return is_task_success(draft, example["expected"])


a2a_rows = []
for run in [f"{A2A_MODEL}|cal21_cat2"] + [f"{A2A_MODEL}|{v}" for v in A2A_VARIANTS]:
    for source in A2A_SOURCES:
        rows = [r for r in parser_results if r["llm_model"] == run and r["source"] == source]
        if not rows:
            continue
        gold_texts = [by_id[r["id"]]["transcript_gold"] for r in rows]
        a2a_rows.append({
            "run": run, "source": source.replace("asr:vinai/PhoWhisper-", ""), "n": len(rows),
            "WER raw": round(sum(word_error_rate(r["transcript"], t) for r, t in zip(rows, gold_texts)) / len(rows), 3),
            "WER corrected": round(sum(word_error_rate(corrected_text(r), t) for r, t in zip(rows, gold_texts)) / len(rows), 3)
            if "a2a" in run else None,
            "title_ok": round(sum(token_f1((r["draft"] or {}).get("title", ""), by_id[r["id"]]["expected"]["title"]) >= 0.5
                                  for r in rows) / len(rows), 3),
            "category_acc": round(sum((r["draft"] or {}).get("category") == by_id[r["id"]]["expected"]["category"]
                                      for r in rows) / len(rows), 3),
            "TSR rule(raw)": round(sum(success_with_rule_deadline(r, r["transcript"]) for r in rows) / len(rows), 3),
            "TSR rule(corrected)": round(sum(success_with_rule_deadline(r, corrected_text(r)) for r in rows) / len(rows), 3),
            "latency_s": round(sum(r["processing_seconds"] for r in rows) / len(rows), 2),
        })
print("A2a: repair the transcript first   (WER corrected < WER raw means the LLM really fixed hearing errors)")
display(pd.DataFrame(a2a_rows))

# paired test on the ASR source: did repair help task success?
asr_source = A2A_SOURCES[-1]
base = {r["id"]: success_with_rule_deadline(r, r["transcript"]) for r in parser_results
        if r["llm_model"] == f"{A2A_MODEL}|cal21_cat2" and r["source"] == asr_source}
for variant_name in A2A_VARIANTS:
    repaired = {r["id"]: success_with_rule_deadline(r, corrected_text(r)) for r in parser_results
                if r["llm_model"] == f"{A2A_MODEL}|{variant_name}" and r["source"] == asr_source}
    if base and repaired:
        print(PairedComparison("cal21_cat2", base, variant_name, repaired).summary())

# read the repairs: which ones are right, which ones are hallucinated?
repairs = [{"id": r["id"], "raw": r["transcript"][:50], "corrected": corrected_text(r)[:50],
            "gold": by_id[r["id"]]["transcript_gold"][:50],
            "WER raw→corr": f'{word_error_rate(r["transcript"], by_id[r["id"]]["transcript_gold"]):.2f}→'
                            f'{word_error_rate(corrected_text(r), by_id[r["id"]]["transcript_gold"]):.2f}'}
           for r in parser_results if r["llm_model"] == f"{A2A_MODEL}|a2a" and r["source"] == asr_source
           and corrected_text(r) != r["transcript"]]
print(f"{len(repairs)} transcripts changed by the LLM (a2a, {asr_source})")
display(pd.DataFrame(repairs))

### 9e. Download results again (includes the experiments above)

In [ ]:
archive_path = shutil.make_archive(str(WORKING_DIR / "voice_eval_outputs"), "zip", OUTPUT_DIR)
print("Download from the Output tab:", archive_path)
for path in sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(f"  {path.name}: {sum(1 for _ in open(path, encoding='utf-8'))} rows")